# Mathematik II/B — Übung 03 (WT 2026)

**Thema:** Kurven und Ableitungen

**Datum:** 01.2026

**Dozent/in:** M.sc. Aigerim Yessim

---

## 📖 Inhaltsverzeichnis

0. [🔄 Wiederholung – Aus Übung 02](#wiederholung)
1. [📖 Theoretischer Hintergrund](#theorie)
2. [📝 Aufgabeblatt 03 – Mathematik II/B Übung 03](#aufgabeblatt)
   - [Aufgabe 3.1](#aufgabe1)
   - [Aufgabe 3.2](#aufgabe2)
   - [Aufgabe 3.3](#aufgabe3)
   - [Aufgabe 3.4](#aufgabe4)
   - [Aufgabe 3.5](#aufgabe5)
   - [Aufgabe 3.6](#aufgabe6)
   - [Aufgabe 3.7](#aufgabe7)
3. [💡 Lösungstipps](#tipps)

---

In [2]:
# Imports
# load the packages you need
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import math
import sympy as sp
import pandas as pd
from IPython.display import display, Math, HTML, Markdown

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, interactive
import ipywidgets as widgets

# Konfiguration für schönere Plots
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✓ Alle Pakete erfolgreich geladen!")

✓ Alle Pakete erfolgreich geladen!


In [3]:
from IPython.display import display, HTML, Markdown
from ipywidgets import VBox, Button, Output

def create_solution_toggle(problem_markdown, solution_markdown, solution_title="Lösung anzeigen", plot_callback=None):
    """
    Erstellt ein interaktives Toggle-Element für Lösungen mit ipywidgets
    
    Args:
        problem_markdown: Aufgabentext (Markdown format for proper LaTeX rendering)
        solution_markdown: Lösungstext (Markdown format for proper LaTeX rendering)
        solution_title: Titel des Lösungsblocks
        plot_callback: Optional function to call when solution is shown (for generating plots)
    """
    # Problem-Container using Output widget
    problem_output = Output()
    with problem_output:
        display(Markdown(f"""
<div style="background-color: #f9f9f9; border-left: 4px solid #0066cc; padding: 20px; margin: 20px 0; border-radius: 5px;">
<h3 style="color: #333;">📝</h3>

{problem_markdown}

</div>
        """))
    
    # Button
    button = Button(description=f"▶ {solution_title}", 
                   button_style='info',
                   tooltip='Klicken zum Anzeigen der Lösung',
                   layout={'width': '200px', 'padding': '10px', 'font_size': '13px'})
    
    # Output für Lösung
    output = Output()
    
    is_shown = [False]  # Mutable flag to track visibility
    
    def toggle_solution(b):
        output.clear_output()
        is_shown[0] = not is_shown[0]
        
        if is_shown[0]:
            button.description = f"▼ Lösung verbergen"
            with output:
                # Use Markdown for proper LaTeX rendering
                display(Markdown(f"""
<div style="margin-top: 15px; background-color: #e8f4f8; padding: 15px; border-radius: 5px; border: 1px solid #00ccff;">

{solution_markdown}

</div>
                """))
                # Call plot callback if provided
                if plot_callback is not None:
                    plot_callback()
        else:
            button.description = f"▶ {solution_title}"
    
    button.on_click(toggle_solution)
    
    return VBox([problem_output, button, output])

#print("✓ Wiederholungs-Widget geladen!")

In [4]:
def plot_function_analysis(
    x_range,
    x_values=None,
    y_values=None,
    func=None,
    maxima=None,
    minima=None,
    nullstellen=None,
    sattelpunkte=None,
    wendepunkte=None,
    asymptoten=None,
    title='Function Analysis',
    x_label='x',
    y_label='f(x)',
    y_limits=None,
    show_inline_labels=True,
    show_grid=True,
    show_legend=True
):
    """
    Visualisiert eine Funktionsanalyse mit automatischer Erkennung und Markierung kritischer Punkte.
    
    Args:
        x_range: (xmin, xmax) Bereich für die x-Achse
        x_values: Optional. Die x-Werte zum Plotten (wenn nicht gegeben, aus func generiert)
        y_values: Optional. Die y-Werte zum Plotten (wenn nicht gegeben, aus func generiert)
        func: Optional. Die Funktion f(x) (wird ignoriert, wenn x_values und y_values gegeben)
        maxima: Liste von (x, y) Maxima
        minima: Liste von (x, y) Minima
        nullstellen: Liste von (x, y) Nullstellen
        sattelpunkte: Liste von (x, y) Sattelpunkte
        wendepunkte: Liste von (x, y) Wendepunkte
        asymptoten: Dict mit Schlüsseln 'vertical', 'horizontal', 'oblique', 'curved'
        title: Titel des Plots
        x_label: Label für x-Achse
        y_label: Label für y-Achse
        y_limits: (ymin, ymax) oder None
        show_inline_labels: Inline-Labels für Punkte anzeigen
        show_grid: Gitter anzeigen
        show_legend: Legende anzeigen
    """
    
    # Generiere x_values und y_values wenn nicht gegeben
    if x_values is None and y_values is None:
        if func is None:
            raise ValueError("Entweder 'func' oder 'x_values' und 'y_values' müssen gegeben sein")
        xmin, xmax = x_range
        x_values = np.linspace(xmin, xmax, 400)
        y_values = func(x_values)
    
    # Default values
    xmin, xmax = x_range
    maxima = maxima or []
    minima = minima or []
    nullstellen = nullstellen or []
    sattelpunkte = sattelpunkte or []
    wendepunkte = wendepunkte or []
    asymptoten = asymptoten or {}
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(x_values, y_values, linewidth=2.5, label='f(x)')
    ax.axhline(0, color='black', linewidth=1.5)
    ax.axvline(0, color='black', linewidth=1.5)

    # Collect all special points for overlap detection
    all_points = []
    if maxima:
        for p in maxima:
            all_points.append((p[0], p[1], 'Max', 'crimson', '^'))
    if minima:
        for p in minima:
            all_points.append((p[0], p[1], 'Min', 'navy', 'v'))
    if nullstellen:
        for p in nullstellen:
            all_points.append((p[0], p[1], 'NS', 'black', 'o'))
    if sattelpunkte:
        for p in sattelpunkte:
            all_points.append((p[0], p[1], 'SP', 'orange', 's'))
    if wendepunkte:
        for p in wendepunkte:
            all_points.append((p[0], p[1], 'WP', 'purple', 'D'))

    # Merge overlapping points (within tolerance)
    tolerance = 0.01 * (xmax - xmin)  # 1% of x-range
    merged_points = []
    used = set()
    
    for i, (x1, y1, label1, color1, marker1) in enumerate(all_points):
        if i in used:
            continue
        labels = [label1]
        colors = [color1]
        marker = 'D'  # Rectangle for merged points
        
        for j, (x2, y2, label2, color2, marker2) in enumerate(all_points):
            if i != j and j not in used:
                if abs(x1 - x2) < tolerance and abs(y1 - y2) < tolerance:
                    labels.append(label2)
                    colors.append(color2)
                    used.add(j)
        
        used.add(i)
        merged_label = '/'.join(labels)
        merged_color = colors[0]  # Use first color
        merged_points.append((x1, y1, merged_label, merged_color, marker))
    
    # Plot merged points
    for px, py, label, color, marker in merged_points:
        ax.scatter([px], [py], s=80, c=color, marker=marker, zorder=5)
        if show_inline_labels:
            ax.text(px, py, f" {label}", fontsize=9, 
                   verticalalignment='bottom', bbox=dict(boxstyle='round', 
                   facecolor='white', alpha=0.8))

    # Asymptotes (dashed lines) with inline labels
    if asymptoten:
        for x0 in asymptoten.get('vertical', []):
            ax.axvline(x0, color='gray', linestyle='--', linewidth=1.5, label=f'x = {x0}')
            if show_inline_labels:
                y_pos = ax.get_ylim()[1] * 0.9 if y_limits is None else y_limits[1] * 0.9
                ax.text(x0, y_pos, f'x={x0}', fontsize=9, color='gray',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        for y0 in asymptoten.get('horizontal', []):
            ax.axhline(y0, color='gray', linestyle='--', linewidth=1.5, label=f'y = {y0}')
            if show_inline_labels:
                x_pos = xmax * 0.85
                ax.text(x_pos, y0, f'y={y0}', fontsize=9, color='gray',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        for line in asymptoten.get('oblique', []):
            if isinstance(line, (list, tuple)) and len(line) == 2:
                m, b = line
                ax.plot(x_values, m * x_values + b, color='gray', linestyle='--', linewidth=1.5, label=f'y = {m}x + {b}')
                if show_inline_labels:
                    x_pos = xmax * 0.85
                    y_pos = m * x_pos + b
                    equation = f'y = {m}x + {b}' if b >= 0 else f'y = {m}x - {abs(b)}'
                    ax.text(x_pos, y_pos, equation, fontsize=9, color='gray',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        for curve in asymptoten.get('curved', []):
            if isinstance(curve, tuple) and len(curve) == 2 and callable(curve[0]):
                # (function, label) format
                func_curve, label_curve = curve
                try:
                    y_curve = func_curve(x_values)
                    ax.plot(x_values, y_curve, color='gray', linestyle='--', linewidth=1.5, label=label_curve)
                    if show_inline_labels:
                        # Place label safely inside the plot area at center
                        x_pos = (xmin + xmax) / 2  # Middle of x-range
                        try:
                            y_pos_val = func_curve(np.array([x_pos]))[0]
                        except:
                            y_pos_val = 0
                        
                        # Clamp to middle 80% of y-range for safety
                        if y_limits is not None:
                            y_min, y_max = y_limits
                            y_center = (y_min + y_max) / 2
                            y_range = (y_max - y_min) * 0.4
                            y_pos_val = np.clip(y_pos_val, y_center - y_range, y_center + y_range)
                        
                        ax.text(x_pos, y_pos_val, label_curve, fontsize=9, color='gray',
                               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                except Exception as e:
                    print(f"Fehler beim Plotten der kurvenförmigen Asymptote: {e}")
            elif callable(curve):
                # Just a function
                try:
                    y_curve = curve(x_values)
                    ax.plot(x_values, y_curve, color='gray', linestyle='--', linewidth=1.5, label='Kurvenförmige Asymptote')
                except Exception as e:
                    print(f"Fehler beim Plotten der kurvenförmigen Asymptote: {e}")

    if show_grid:
        ax.grid(True, alpha=0.3)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(x_label, fontsize=11, fontweight='bold')
    ax.set_ylabel(y_label, fontsize=11, fontweight='bold')
    if y_limits is not None:
        ax.set_ylim(*y_limits)
    if show_legend:
        ax.legend(fontsize=10)
    plt.subplots_adjust(bottom=0.15, left=0.12, right=0.95, top=0.93)
    plt.show()

def plot_convergence(
    n_values,
    a_n,
    limit,
    n_max,
    distance=None,
    epsilon=0.01,
    yscale='log',
    title='Konvergenz einer Folge'
):
    """
    Visualisiert die Konvergenz einer Folge bis zu einem bestimmten n_max.
    
    Args:
        n_values: Array der Indizes n
        a_n: Array der Folgenglieder
        limit: Der Grenzwert L
        n_max: Maximaler n-Wert der angezeigt werden soll
        distance: Optional. Array der Abstände |a_n - L| (wird berechnet wenn nicht gegeben)
        epsilon: Toleranz-Schwellenwert
        yscale: 'linear' oder 'log'
        title: Titel des Plots
    """
    if distance is None:
        distance = np.abs(a_n - limit)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Folgenglieder und Grenzwert
    mask = n_values <= n_max
    ax1.plot(n_values[mask], a_n[mask], 'b.-', markersize=6, label=f'$a_n$')
    ax1.axhline(limit, color='red', linestyle='--', linewidth=2, label=f'Grenzwert L = {limit}')
    ax1.fill_between(n_values[mask], limit - epsilon, limit + epsilon, alpha=0.2, color='green', label=f'±ε = ±{epsilon}')
    ax1.set_xlabel('n', fontsize=11)
    ax1.set_ylabel('$a_n$', fontsize=11)
    ax1.set_title(title, fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(1, max(n_values) * 1.05)
    
    # Plot 2: Abstand zum Grenzwert (logarithmisch)
    ax2.semilogy(n_values[mask], distance[mask], 'g.-', markersize=6, label='$|a_n - L|$')
    ax2.axhline(epsilon, color='red', linestyle='--', linewidth=2, label=f'ε = {epsilon}')
    ax2.set_xlabel('n', fontsize=11)
    ax2.set_ylabel('$|a_n - L|$', fontsize=11)
    ax2.set_title('Konvergenzgeschwindigkeit (logarithmisch)', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3, which='both')
    ax2.set_xlim(1, max(n_values) * 1.05)
    
    plt.tight_layout()
    plt.show()
    
    # Ausgabe von Konvergenzstatistik
    epsilon_reached = np.where(distance <= epsilon)[0]
    if len(epsilon_reached) > 0:
        n_critical = n_values[epsilon_reached[0]]
        print(f"✓ Ab n ≥ {n_critical} ist |a_n - L| < ε = {epsilon}")

In [5]:
def plot_parametric_curve_2d(
    r_func,
    t_range,
    eval_points=None,
    show_tangent=True,
    show_normal=True,
    show_second_derivative=True,
    vector_scale=1.0,
    num_points=400,
    title='Parametrische Kurve',
    x_label='x',
    y_label='y',
    curve_color='blue',
    tangent_color='green',
    normal_color='red',
    second_deriv_color='red',
    figsize=(10, 8)
):
    """
    Visualisiert eine parametrische 2D-Kurve mit Tangenten-, Normalen- und zweiten Ableitungsvektoren.
    
    Args:
        r_func: Funktion r(t) die einen Tupel/Array (x(t), y(t)) zurückgibt
        t_range: (t_min, t_max) Bereich für den Parameter t
        eval_points: Liste von t-Werten, an denen Tangenten/Normalen gezeigt werden sollen
        show_tangent: Tangentenvektoren anzeigen
        show_normal: Normalenvektoren anzeigen
        show_second_derivative: Zweite Ableitungsvektoren anzeigen (als gestrichelte rote Linie)
        vector_scale: Skalierungsfaktor für die Vektoren
        num_points: Anzahl der Punkte für die Kurve
        title: Titel des Plots
        x_label: Label für x-Achse
        y_label: Label für y-Achse
        curve_color: Farbe der Kurve
        tangent_color: Farbe der Tangentenvektoren
        normal_color: Farbe der Normalenvektoren
        second_deriv_color: Farbe der zweiten Ableitungsvektoren
        figsize: Größe der Figur
    
    Beispiel:
        def r(t):
            return (np.abs(t), t**3)
        
        plot_parametric_curve_2d(r, (-3, 3), eval_points=[2], vector_scale=0.5)
    """
    
    # Erzeuge t-Werte für die Kurve
    t_min, t_max = t_range
    t_values = np.linspace(t_min, t_max, num_points)
    
    # Berechne Kurvenpunkte
    curve_points = np.array([r_func(t) for t in t_values])
    x_curve = curve_points[:, 0]
    y_curve = curve_points[:, 1]
    
    # Erstelle Plot
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plotte die Kurve
    ax.plot(x_curve, y_curve, color=curve_color, linewidth=2.5, label='$r(t)$', zorder=1)
    
    # Wenn Evaluationspunkte gegeben sind
    if eval_points:
        for t_eval in eval_points:
            # Berechne Position auf der Kurve
            r_t = np.array(r_func(t_eval))
            x_t, y_t = r_t
            
            # Markiere den Punkt auf der Kurve
            ax.scatter([x_t], [y_t], s=100, c='black', marker='o', zorder=5, 
                      label=f'$r({t_eval})$' if t_eval == eval_points[0] else '')
            
            # Berechne numerische Ableitung (Tangentenvektor)
            h = 0.001
            r_t_plus = np.array(r_func(t_eval + h))
            r_t_minus = np.array(r_func(t_eval - h))
            dr_dt = (r_t_plus - r_t_minus) / (2 * h)
            
            # Normalisiere und skaliere den Tangentenvektor
            tangent_length = np.linalg.norm(dr_dt)
            if tangent_length > 0:
                dr_dt_normalized = dr_dt / tangent_length * vector_scale
            else:
                dr_dt_normalized = dr_dt
            
            # Berechne Normalenvektor (90° Drehung im Uhrzeigersinn: (x,y) -> (y,-x))
            # oder gegen Uhrzeigersinn: (x,y) -> (-y,x)
            normal_vec = np.array([-dr_dt_normalized[1], dr_dt_normalized[0]])
            
            # Berechne numerische zweite Ableitung
            h = 0.001
            r_t = np.array(r_func(t_eval))
            r_t_plus_2h = np.array(r_func(t_eval + 2*h))
            r_t_plus = np.array(r_func(t_eval + h))
            r_t_minus = np.array(r_func(t_eval - h))
            r_t_minus_2h = np.array(r_func(t_eval - 2*h))
            
            # Zweite Ableitung: r''(t) ≈ (r(t+2h) - 2*r(t) + r(t-2h)) / (4h²)
            d2r_dt2 = (r_t_plus_2h - 2*r_t + r_t_minus_2h) / (4 * h**2)
            
            # Normalisiere und skaliere den zweiten Ableitungsvektor
            second_deriv_length = np.linalg.norm(d2r_dt2)
            if second_deriv_length > 0:
                d2r_dt2_normalized = d2r_dt2 / second_deriv_length * vector_scale
            else:
                d2r_dt2_normalized = d2r_dt2
            
            # Plotte Tangentenvektor
            if show_tangent:
                ax.arrow(x_t, y_t, dr_dt_normalized[0], dr_dt_normalized[1],
                        head_width=0.15, head_length=0.1, fc=tangent_color, 
                        ec=tangent_color, linewidth=2, zorder=4,
                        label='Tangentenvektor' if t_eval == eval_points[0] else '')
                # Beschriftung für Tangentenvektor
                ax.text(x_t + dr_dt_normalized[0] * 1.2, y_t + dr_dt_normalized[1] * 1.2,
                       f"$r'({t_eval})$", fontsize=10, color=tangent_color,
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
            
            # Plotte Normalenvektor
            if show_normal:
                ax.arrow(x_t, y_t, normal_vec[0], normal_vec[1],
                        head_width=0.15, head_length=0.1, fc=normal_color, 
                        ec=normal_color, linewidth=2, zorder=4,
                        label='Normalenvektor' if t_eval == eval_points[0] else '')
                # Beschriftung für Normalenvektor
                ax.text(x_t + normal_vec[0] * 1.2, y_t + normal_vec[1] * 1.2,
                       f"$n({t_eval})$", fontsize=10, color=normal_color,
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
            
            # Plotte zweiten Ableitungsvektor (als gestrichelte Linie)
            if show_second_derivative:
                ax.arrow(x_t, y_t, d2r_dt2_normalized[0], d2r_dt2_normalized[1],
                        head_width=0.15, head_length=0.1, fc=second_deriv_color, 
                        ec=second_deriv_color, linewidth=2, linestyle='--', zorder=4,
                        label='Zweite Ableitung' if t_eval == eval_points[0] else '')
                # Beschriftung für zweiten Ableitungsvektor
                ax.text(x_t + d2r_dt2_normalized[0] * 1.2, y_t + d2r_dt2_normalized[1] * 1.2,
                       f"$r''({t_eval})$", fontsize=10, color=second_deriv_color,
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
            
            # Zeige t-Wert am Punkt
            ax.text(x_t, y_t - 0.3, f't={t_eval}', fontsize=9, ha='center',
                   bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.6))
    
    # Formatierung
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.3)
    ax.axvline(0, color='black', linewidth=0.8, alpha=0.3)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel(x_label, fontsize=11, fontweight='bold')
    ax.set_ylabel(y_label, fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_aspect('equal', adjustable='box')
    ax.legend(fontsize=10, loc='best')
    
    plt.tight_layout()
    plt.show()
    
    # Ausgabe von Informationen
    if eval_points:
        print("=" * 60)
        print("Informationen zu den Evaluationspunkten:")
        print("=" * 60)
        for t_eval in eval_points:
            r_t = np.array(r_func(t_eval))
            h = 0.001
            r_t_plus = np.array(r_func(t_eval + h))
            r_t_minus = np.array(r_func(t_eval - h))
            dr_dt = (r_t_plus - r_t_minus) / (2 * h)
            
            # Zweite Ableitung
            r_t_plus_2h = np.array(r_func(t_eval + 2*h))
            r_t_minus_2h = np.array(r_func(t_eval - 2*h))
            d2r_dt2 = (r_t_plus_2h - 2*r_t + r_t_minus_2h) / (4 * h**2)
            
            normal_vec = np.array([-dr_dt[1], dr_dt[0]])
            
            print(f"\nBei t = {t_eval}:")
            print(f"  Position r({t_eval}) = ({r_t[0]:.4f}, {r_t[1]:.4f})")
            print(f"  Tangentenvektor r'({t_eval}) ≈ ({dr_dt[0]:.4f}, {dr_dt[1]:.4f})")
            print(f"  Normalenvektor n({t_eval}) ≈ ({normal_vec[0]:.4f}, {normal_vec[1]:.4f})")
            print(f"  Zweite Ableitung r''({t_eval}) ≈ ({d2r_dt2[0]:.4f}, {d2r_dt2[1]:.4f})")
            print(f"  |r'({t_eval})| ≈ {np.linalg.norm(dr_dt):.4f}")
            print(f"  |r''({t_eval})| ≈ {np.linalg.norm(d2r_dt2):.4f}")
        print("=" * 60)

def make_interactive_slider(plot_func, param_name, param_values, **fixed_kwargs):
    """
    Macht eine Plot-Funktion interaktiv mit einem Schieberegler.
    
    Args:
        plot_func: Die Plot-Funktion, die aufgerufen werden soll
        param_name: Name des Parameters, der variiert werden soll (z.B. 'eval_points')
        param_values: Liste der möglichen Werte für den Parameter
        **fixed_kwargs: Alle anderen festen Parameter für die Plot-Funktion
    
    Beispiel:
        # Macht plot_parametric_curve_2d interaktiv für verschiedene t-Werte
        make_interactive_slider(
            plot_parametric_curve_2d,
            param_name='eval_points',
            param_values=[0, 1, 2, 3, 4],
            r_func=my_curve,
            t_range=(-5, 5),
            vector_scale=0.5
        )
    """
    from ipywidgets import FloatSlider, interact
    
    def wrapper(param_val):
        """Wrapper-Funktion die den variablen Parameter setzt"""
        # Setze den variablen Parameter
        kwargs = fixed_kwargs.copy()
        kwargs[param_name] = [param_val]  # Als Liste für eval_points
        
        # Rufe die Original-Plot-Funktion auf
        plot_func(**kwargs)
    
    # Bestimme min/max und step für den Slider
    param_min = min(param_values)
    param_max = max(param_values)
    
    if len(param_values) > 1:
        # Finde die kleinste Differenz zwischen aufeinanderfolgenden Werten
        diffs = [abs(param_values[i+1] - param_values[i]) for i in range(len(param_values)-1)]
        step = min(diffs)
    else:
        step = 0.1
    
    # Erstelle Slider
    slider = FloatSlider(
        value=param_values[0] if len(param_values) > 0 else 0,
        min=param_min,
        max=param_max,
        step=step,
        description=f'{param_name}:',
        style={'description_width': '100px'},
        layout={'width': '500px'}
    )
    
    # Erstelle interaktiven Plot
    interact(wrapper, param_val=slider)

In [6]:
print("=" * 80)
print("📚 WIEDERHOLUNG : Ableitungen (Aus Übung 2, Aufgabe 2.6)")
print("=" * 80)


problem_2_1 = r"""
 "<a id='wiederholung'></a>"
## 🔄 Wiederholung – Aus Übung 02

Zur Auffrischung: In Übung 02 haben Sie unter anderem gelernt, wie man verschiedene Funktionen auf **Stetigkeit prüft** und **ableitet**.
Hier sind **Selbstkontrolle-Aufgaben** aus Übung 02.6 – versuchen Sie diese selbstständig zu lösen, bevor Sie die Lösung anschauen.

---

Berechnen Sie die **erste Ableitung** der folgenden Funktionen:

## Aufgabe 2.3: Ableitungen

Berechnen Sie die erste Ableitung der folgenden Funktionen:

a) $f_{13}(t) = \sin^3\!\left(e^{2t^2} + t^5\right)$

b) $f_{14}(t) = \sqrt{2t^2 + 1}$

c) $f_{16}(t) = \ln(t^2) - \ln(t^5)$

**Versuchen Sie allein zu lösen**
"""


solution_2_1 = r"""
### Lösungen – Aufgabe 2.6

---

#### a) $f_{13}(t) = \sin^3\!\left(e^{2t^2} + t^5\right)$

**Regel: Kettenregel (zweifach)**

$$u = e^{2t^2} + t^5 \quad \Rightarrow \quad u' = 4t\,e^{2t^2} + 5t^4$$

$$\boxed{f'_{13}(t) = 3\sin^2\!\left(e^{2t^2}+t^5\right)\cos\!\left(e^{2t^2}+t^5\right)\cdot\left(4t\,e^{2t^2}+5t^4\right)}$$

---

#### b) $f_{14}(t) = \sqrt{2t^2 + 1}$

**Regel: Kettenregel**

$$f_{14}(t) = (2t^2+1)^{1/2} \quad \Rightarrow \quad u = 2t^2+1,\quad u' = 4t$$

$$\boxed{f'_{14}(t) = \frac{4t}{2\sqrt{2t^2+1}} = \frac{2t}{\sqrt{2t^2+1}}}$$

---

#### c) $f_{16}(t) = \ln(t^2) - \ln(t^5)$

**Regel: Logarithmusgesetze + Kettenregel**

$$f_{16}(t) = 2\ln(t) - 5\ln(t)= -3\ln(t)$$

$$\boxed{f'_{16}(t) = -\frac{3}{t}}$$

**Hier sieht man wie diese Ableitung immer positiv bleibt. Ein Beweis dafür , dass unsere Funktion monoton steigend ist**
"""

def plot_wiederholung_2():
    import numpy as np
    import matplotlib.pyplot as plt

    t = np.linspace(0.01, 3, 500)

    f14       = np.sqrt(2*t**2 + 1)
    f14_prime = 2*t / np.sqrt(2*t**2 + 1)

    plt.figure(figsize=(9, 5))
    plt.plot(t, f14,       label=r"$f_{14}(t) = \sqrt{2t^2+1}$", color="royalblue", linewidth=2)
    plt.plot(t, f14_prime, label=r"$f'_{14}(t) = \dfrac{2t}{\sqrt{2t^2+1}}$", color="tomato", linewidth=2, linestyle="--")

    plt.axhline(0, color="black", linewidth=0.8)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.title(r"$f_{14}$ und $f'_{14}$", fontsize=14)
    plt.xlabel("t")
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()






display(create_solution_toggle(problem_2_1, solution_2_1, "Lösung anzeigen", plot_wiederholung_2))


📚 WIEDERHOLUNG : Ableitungen (Aus Übung 2, Aufgabe 2.6)


<a id="theorie"></a>
## 📖 Theoretischer Hintergrund

# Mathematik II/B – Blatt 3: Kurven und Ableitungen

---

## Parametrisierte Kurven und Tangentenvektor

Eine parametrisierte Kurve ist eine Funktion

$$r: I \to \mathbb{R}^n, \quad t \mapsto r(t),$$

wobei $I \subset \mathbb{R}$ ein Intervall ist und $r(t)$ die Position eines Punktes im $\mathbb{R}^n$ beschreibt.  
In Komponenten schreibt man

$$r(t) = \begin{pmatrix} x_1(t) \\ x_2(t) \\ \vdots \\ x_n(t) \end{pmatrix}.$$

Der Parameter $t$ kann als „Zeit" interpretiert werden: Für jedes $t \in I$ liefert $r(t)$ einen Punkt im Raum.  
Die **Spurkurve** (oder das Bild der Kurve) ist die Punktmenge

$$\{r(t) \mid t \in I\} \subset \mathbb{R}^n.$$

> **Wichtig:** Verschiedene Parametrisierungen können dieselbe Spurkurve beschreiben.

---

## Tangentenvektor an die Kurve

Ist $r$ differenzierbar, dann heißt

$$r'(t) = \frac{d}{dt} r(t)$$

der **Tangentenvektor** (auch Geschwindigkeitsvektor) im Punkt $r(t)$.  
Er zeigt in die Richtung, in der die Kurve für wachsendes $t$ weiterläuft.  
In der Ebene ($n = 2$) gilt:

$$r(t) = \begin{pmatrix} x(t) \\ y(t) \end{pmatrix} \implies r'(t) = \begin{pmatrix} x'(t) \\ y'(t) \end{pmatrix}.$$

Die **Tangente** an die Kurve im Zeitpunkt $t_0$ in Parameterform:

$$\ell(s) = r(t_0) + s\, r'(t_0), \quad s \in \mathbb{R}.$$

---

## Parameterelimination: Von der Parametrisierung zur Gleichung in $x$ und $y$

Um die Spurkurve ohne Parameter darzustellen, leitet man aus den beiden Gleichungen  
$x = x(t)$ und $y = y(t)$ eine Beziehung her, die nur noch $x$ und $y$ enthält.

### Vorgehensweise (Rezept):

1. **Geeignete Gleichung wählen:** Versuchen Sie, aus $x = x(t)$ oder $y = y(t)$ den Parameter $t$ auszudrücken.

2. **Einsetzen:** Setzen Sie den Ausdruck für $t$ in die andere Komponente ein und erhalten Sie eine Gleichung $F(x,y) = 0$ oder (falls möglich) $y = y(x)$.

3. **Existenzbedingungen notieren:**
   - Bedingungen aus dem Intervall $t \in I$ (Start-/Endwerte!)
   - Bedingungen aus Umformungen (z.B. Wurzeln: Radikand $\geq 0$)
   - Falls $y = y(x)$ verlangt ist: Prüfen, ob zu einem $x$ evtl. mehrere $y$-Werte gehören (dann hat die Spurkurve mehrere Zweige).

4. **Skizze:** Markieren Sie wichtige Punkte (z.B. $t$-Randwerte, Achsenschnitte, Scheitelpunkte) und geben Sie die Bewegungsrichtung (wachsendes $t$) an.

---

# Beispiel 1: Parabel durch Parameterelimination

Gegeben sei die parametrisierte Kurve

$$
r(t) =
\begin{pmatrix}
x(t) \\
y(t)
\end{pmatrix}
=
\begin{pmatrix}
2t - 1 \\
t^2
\end{pmatrix},
\quad 0 \le t \le 3.
$$

Eine Parametrisierung liefert zu jedem Parameterwert $t$ einen Punkt $(x(t), y(t))$.  
Wenn man den Graphen der Kurve als Beziehung zwischen $x$ und $y$ angeben möchte (also ohne Parameter), muss man den Parameter $t$ eliminieren.

Das bedeutet:  
Wir wollen eine Gleichung finden, die genau von den Punkten $(x,y)$ erfüllt wird, die auf der Kurve liegen.

---

## Idee

Man drückt $t$ aus einer der beiden Gleichungen  
$x = x(t)$ oder $y = y(t)$ aus  
und setzt diesen Ausdruck in die andere Gleichung ein.

So verschwindet $t$ und es entsteht eine Beziehung zwischen $x$ und $y$.

---

## Schritt 1: Parameter ausdrücken

Wir verwenden die Gleichung

$$
x = 2t - 1,
$$

weil sie sich leicht nach $t$ umstellen lässt.

Daraus folgt

$$
t = \frac{x + 1}{2}.
$$

---

## Schritt 2: Einsetzen (Parameter eliminieren)

Nun setzen wir den Ausdruck für $t$ in die zweite Gleichung  
$y = t^2$ ein:

$$
y = \left( \frac{x + 1}{2} \right)^2.
$$

Damit haben wir die Kurve als Funktionsgleichung $y = y(x)$ beschrieben.

---

## Schritt 3: Bereich berücksichtigen

Die Gleichung

$$
y = \left( \frac{x + 1}{2} \right)^2
$$

beschreibt die gesamte Parabel.

Unsere Parametrisierung ist jedoch nur für  
$0 \le t \le 3$ definiert.

Deshalb übertragen wir die Einschränkung an $t$ auf die möglichen $x$-Werte:

$$
x(0) = 2 \cdot 0 - 1 = -1,
$$

$$
x(3) = 2 \cdot 3 - 1 = 5.
$$

Also gilt für den beschriebenen Kurvenabschnitt:

$$
-1 \le x \le 5.
$$

Entsprechend folgt

$$
0 \le y \le 9,
$$

da $y = t^2$ auf dem Intervall $[0,3]$ Werte zwischen 0 und 9 annimmt.

---

# Ergebnis

Die Spurkurve ist der Parabelabschnitt

$$
y = \left( \frac{x + 1}{2} \right)^2,
\quad -1 \le x \le 5.
$$
---

---

# Beispiel 2: Ellipse $$\frac{x^2}{a^2} + \frac{y^2}{b^2} = 1$$ durch Parameterdarstellung

Hier geht es um die umgekehrte Richtung wie beim Parabelbeispiel:  
Wir starten mit einer Gleichung der Kurve (Ellipse) und geben eine Parametrisierung an, die diese Kurve beschreibt.

**Ziel:**  
Wir wollen eine Funktion  

$$
r : [0, 2\pi) \to \mathbb{R}^2
$$

angeben, so dass $r(t)$ genau alle Punkte der Ellipse durchläuft.

---

## Idee

Wir benutzen den Einheitskreis

$$
\cos^2 t + \sin^2 t = 1.
$$

Da die Ellipse im Vergleich zum Einheitskreis  

- in $x$‑Richtung mit Faktor $a$  
- in $y$‑Richtung mit Faktor $b$  

gestreckt ist, skalieren wir die Kreisparametrisierung entsprechend.

---

## Schritt 1: Parametrisierung ansetzen

Wir setzen

$$
r(t) =
\begin{pmatrix}
x(t) \\
y(t)
\end{pmatrix}
=
\begin{pmatrix}
a \cos t \\
b \sin t
\end{pmatrix},
\quad 0 \le t < 2\pi.
$$

Damit gilt automatisch:

$$
-a \le x(t) \le a,
\qquad
-b \le y(t) \le b.
$$

---

## Schritt 2: Prüfen, dass die Punkte auf der Ellipse liegen

Wir setzen $x(t)$ und $y(t)$ in die Ellipsengleichung ein:

$$
\frac{x(t)^2}{a^2}
=
\frac{a^2 \cos^2 t}{a^2}
=
\cos^2 t,
$$

$$
\frac{y(t)^2}{b^2}
=
\frac{b^2 \sin^2 t}{b^2}
=
\sin^2 t.
$$

Addieren liefert:

$$
\frac{x(t)^2}{a^2} + \frac{y(t)^2}{b^2}
=
\cos^2 t + \sin^2 t
=
1.
$$

Damit liegt jeder Punkt $r(t)$ tatsächlich auf der Ellipse.



# Hinweis zum Parameterintervall $[0,2\pi)$ vs. $[0,2\pi]$

Da Cosinus und Sinus $2\pi$‑periodisch sind, gilt:

$$
r(0) =
\begin{pmatrix}
a \\
0
\end{pmatrix},
\qquad
r(2\pi) =
\begin{pmatrix}
a \\
0
\end{pmatrix}.
$$

Also ist $r(0) = r(2\pi)$.

---

### Wahl des Intervalls

- Wählt man $t \in [0,2\pi)$,  
  dann wird jeder Punkt der Ellipse genau einmal durchlaufen;  
  der Startpunkt bei $t = 0$ wird am Ende nicht nochmals erreicht.

- Wählt man $t \in [0,2\pi]$,  
  dann ist das ebenfalls korrekt,  
  aber der Startpunkt wird bei $t = 2\pi$ ein zweites Mal getroffen.

Wenn man möchte, dass jeder Punkt genau einmal durchlaufen wird,  
ist das Intervall $[0,2\pi)$ die passendere Wahl.

---

# Wertebereiche

Aus

$$
|\cos t| \le 1,
\qquad
|\sin t| \le 1
$$

folgt

$$
-a \le x(t) \le a,
\qquad
-b \le y(t) \le b.
$$

---

# Ergebnis

Die Parametrisierung

$$
r(t) =
\begin{pmatrix}
a \cos t \\
b \sin t
\end{pmatrix},
\quad 0 \le t < 2\pi
$$

beschreibt genau die Ellipse

$$
\frac{x^2}{a^2} + \frac{y^2}{b^2} = 1.
$$



<a id="aufgabeblatt"></a>
## 📝 Aufgabeblatt 03 – Mathematik II/B Übung 03



---

In [7]:


problem_3_1_b = r"""
<a id="aufgabe1"></a>
#### Aufgabe 3.1 b)

Gegeben sei die Kurve:

$$r(t) = \begin{pmatrix} |t| \\ t^3 \end{pmatrix}$$

Prüfen Sie, ob der Normalenvektor für alle $t \in \mathbb{R}$ existiert, 
und berechnen Sie den Normalenvektor im Punkt $t_0 = 2$.

**Versuchen Sie erstmal allein**     
<a href="#tipps">💡Lösungstipps</a>
"""


solution_3_1_b = r"""
#### Lösung 3.1 b)

Die Kurve ist durch ihre Parameterdarstellung gegeben:

$$r(t) = \begin{pmatrix} |t| \\ t^3 \end{pmatrix}$$

Der Tangentenvektor ergibt sich durch die Ableitung von $r(t)$. Da $x(t) = |t|$, 
beachten wir, dass die Ableitung stückweise definiert ist:

$$x'(t) = \begin{cases} 1, & t > 0, \\ -1, & t < 0, \\ \text{nicht definiert,} & t = 0. \end{cases}$$

Die Ableitung von $y(t) = t^3$ ist:

$$y'(t) = 3t^2.$$

Der Tangentialvektor ist somit:

$$r'(t) = \begin{pmatrix} x'(t) \\ y'(t) \end{pmatrix} = \begin{pmatrix} \text{sgn}(t) \\ 3t^2 \end{pmatrix}, \quad t \neq 0.$$

Für $t = 0$ ist $x'(t)$ nicht definiert, daher existiert der Normalenvektor an dieser 
Stelle nicht. Für $t \neq 0$ gilt die Orthogonalitätsbedingung:

$$\langle n(t), r'(t) \rangle = n_1 x'(t) + n_2 y'(t) = 0.$$

Setzen wir $x'(t) = \text{sgn}(t)$ und $y'(t) = 3t^2$, ergibt sich:

$$n_1 \cdot \text{sgn}(t) + n_2 \cdot 3t^2 = 0.$$

Eine mögliche Lösung ist:

$$n_1 = -3t^2, \quad n_2 = \text{sgn}(t).$$

Somit ist der Normalenvektor:

$$n(t) = \begin{pmatrix} -3t^2 \\ \text{sgn}(t) \end{pmatrix}.$$

Für $t_0 = 2$ gilt $x'(2) = 1$ und $y'(2) = 12$. Der Normalenvektor ist:

$$\boxed{n(2) = \begin{pmatrix} -12 \\ 1 \end{pmatrix}.}$$

Der Normalenvektor existiert nicht für $t = 0$, ist jedoch für $t \neq 0$ definiert.
"""



def plot_curve_3_1_b(t0):
    """
    Interaktiver Plot der Kurve r(t) = (|t|, t³)
    mit Tangenten- und Normalenvektor bei t = t0.
    """
    # --- Kurve generieren ---
    t_vals = np.linspace(-3, 3, 500)
    x_vals = np.abs(t_vals)
    y_vals = t_vals**3

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(x_vals, y_vals, 'b-', linewidth=2, label=r'$r(t) = (|t|,\, t^3)$')

    # --- Punkt auf der Kurve ---
    P = np.array([np.abs(t0), t0**3])
    ax.plot(*P, 'ko', markersize=8, zorder=5)
    ax.annotate(f'$r(t_0)$\n$t_0={t0:.2f}$',
                xy=P, xytext=(P[0]+0.15, P[1]+0.3),
                fontsize=10, color='black')

    if np.isclose(t0, 0.0, atol=1e-2):
        # --- Fehlerfall t = 0 ---
        ax.set_title(
            r'$r(t) = (|t|,\, t^3)$' + '\n'
            r'⚠ Tangenten- und Normalenvektor existieren nicht für $t = 0$',
            fontsize=12, color='red'
        )
        ax.text(0.5, 0.02,
                "Fehler: Der Normalenvektor existiert nicht für $t = 0$,\n"
                "da $|t|$ bei $t = 0$ nicht differenzierbar ist.",
                transform=ax.transAxes,
                fontsize=11, color='red',
                ha='center', va='bottom',
                bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.8))
    else:
        # --- Tangentenvektor ---
        sgn = 1.0 if t0 > 0 else -1.0
        T = np.array([sgn, 3 * t0**2])
        T_norm = T / np.linalg.norm(T)

        # --- Normalenvektor ---
        N = np.array([-3 * t0**2, sgn])
        N_norm = N / np.linalg.norm(N)

        scale = 0.6

        ax.annotate('', xy=P + scale * T_norm, xytext=P,
                    arrowprops=dict(arrowstyle='->', color='green', lw=2))
        ax.text(*(P + scale * T_norm + np.array([0.05, 0.1])),
                r"$r'(t_0)$", color='green', fontsize=11)

        ax.annotate('', xy=P + scale * N_norm, xytext=P,
                    arrowprops=dict(arrowstyle='->', color='red', lw=2))
        ax.text(*(P + scale * N_norm + np.array([0.05, 0.1])),
                r"$n(t_0)$", color='red', fontsize=11)

        ax.set_title(
            r'$r(t) = (|t|,\, t^3)$'
            + f'\n$t_0 = {t0:.2f}$'
            + r',  $r\'(t_0) = $'
            + f'$({sgn:.0f},\\ {3*t0**2:.2f})^T$'
            + r',  $n(t_0) = $'
            + f'$({-3*t0**2:.2f},\\ {sgn:.0f})^T$',
            fontsize=11
        )

    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.axvline(0, color='gray', lw=0.8, ls='--')
    ax.set_xlim(-0.3, 3.5)
    ax.set_ylim(-28, 28)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

display(create_solution_toggle(
    problem_3_1_b, 
    solution_3_1_b, 
    "Lösung anzeigen", 
    lambda: interact(
        plot_curve_3_1_b,
        t0=FloatSlider(
            value=2.0,
            min=-3.0,
            max=3.0,
            step=0.1,
            description='$t_0$',
            continuous_update=True,
            style={'description_width': 'initial'},
            layout={'width': '500px'}
        )
    )
))



In [8]:
problem_3_2 = r"""
<a id="aufgabe2"></a>
#### Aufgabe 3.2: Konstante Geschwindigkeit und Orthogonalität

---

**a)** Sei $r : I \to \mathbb{R}^2$ eine zweimal stetig differenzierbare, reguläre Kurve (also $r'(t) \neq 0$).  
Die (Betrags-)Geschwindigkeit sei

$$v(t) = \|r'(t)\|.$$

Zeigen Sie:

$$\frac{d}{dt}\left(v(t)^2\right) = 2\langle r'(t),\, r''(t)\rangle.$$

Folgern Sie daraus, dass $r''(t) \perp r'(t)$ genau dann gilt, wenn $v(t)$ konstant ist.

---

**b)** Betrachten Sie die Kreisparametrisierung

$$r_1(t) = \begin{pmatrix} \cos t \\ \sin t \end{pmatrix}.$$

Berechnen Sie $r_1'(t)$, $r_1''(t)$ und $v_1(t)$.  
Prüfen Sie, ob $\langle r_1'(t),\, r_1''(t)\rangle = 0$ gilt.

---

**c) Umparametrisierung (gleiche Spur, anderes Bewegungsverhalten).**  
Betrachten Sie dieselbe *geometrische Kurve* (denselben Kreis als Punktmenge im $\mathbb{R}^2$),  
jedoch mit einer anderen Parametrisierung:

$$r_2(t) = \begin{pmatrix} \cos(t^3) \\ \sin(t^3) \end{pmatrix}.$$

Diese Umparametrisierung *ändert die Spurkurve nicht*: Als Graph/Menge der Raumpunkte  
bleibt es derselbe Kreis. Aus physikalischer Sicht beschreibt $r_2$ jedoch eine *andere Bewegung*,  
weil die einzelnen Raumpunkte zu *anderen Zeitpunkten* erreicht werden.  
Anders formuliert: Geometrisch ist es dieselbe Kurve, physikalisch aber eine andere Bewegung,  
da der Massenpunkt die Bahn mit einer nicht konstanten Geschwindigkeit durchläuft.

Berechnen Sie $r_2'(t)$, $r_2''(t)$ und $v_2(t)$.  
Prüfen Sie, ob $r_2''(t) \perp r_2'(t)$ für $t \neq 0$ gilt.

---

**d)** Betrachten Sie die Parabel

$$r_3(t) = \begin{pmatrix} t \\ t^2 \end{pmatrix}.$$

Berechnen Sie $r_3'(t)$, $r_3''(t)$ und $v_3(t)$.  
Untersuchen Sie, für welche $t$ (falls überhaupt) gilt:

$$\langle r_3'(t),\, r_3''(t)\rangle = 0.$$


**Versuchen Sie erstmal allein**     
<a href="#tipps">💡Lösungstipps</a>
"""


solution_3_2 = r"""
#### Lösung 3.2

---

**a)** Es gilt

$$v(t)^2 = \|r'(t)\|^2 = \langle r'(t),\, r'(t)\rangle.$$

Differenzieren liefert:

$$\frac{d}{dt}\left(v(t)^2\right) = \frac{d}{dt}\langle r'(t),\, r'(t)\rangle = \langle r''(t),\, r'(t)\rangle + \langle r'(t),\, r''(t)\rangle = 2\langle r'(t),\, r''(t)\rangle.$$

Damit ist

$$\langle r'(t),\, r''(t)\rangle = 0 \iff \frac{d}{dt}\left(v(t)^2\right) = 0 \iff v(t) \text{ ist konstant.}$$

Also ist $r''(t)$ genau dann orthogonal zu $r'(t)$, wenn die Kurve mit konstanter Geschwindigkeit durchlaufen wird.

---

**b)** Ableiten ergibt:

$$r_1'(t) = \begin{pmatrix} -\sin t \\ \cos t \end{pmatrix}, \qquad r_1''(t) = \begin{pmatrix} -\cos t \\ -\sin t \end{pmatrix}.$$

Die Geschwindigkeit ist

$$v_1(t) = \|r_1'(t)\| = \sqrt{\sin^2 t + \cos^2 t} = 1,$$

also konstant. Außerdem:

$$\langle r_1'(t),\, r_1''(t)\rangle = (-\sin t)(-\cos t) + (\cos t)(-\sin t) = \sin t\cos t - \sin t\cos t = 0.$$

---

**c)** Setze $\varphi(t) = t^3$. Dann ist

$$r_2'(t) = \varphi'(t)\begin{pmatrix} -\sin(\varphi(t)) \\ \cos(\varphi(t)) \end{pmatrix} = 3t^2 \begin{pmatrix} -\sin(t^3) \\ \cos(t^3) \end{pmatrix}.$$

Somit

$$v_2(t) = \|r_2'(t)\| = 3t^2\sqrt{\sin^2(t^3) + \cos^2(t^3)} = 3t^2,$$

also nicht konstant (für $t \neq 0$). Mit Teil **(a)** folgt:

$$\langle r_2'(t),\, r_2''(t)\rangle = \frac{1}{2}\frac{d}{dt}\left(v_2(t)^2\right) = \frac{1}{2}\frac{d}{dt}\left(9t^4\right) = \frac{1}{2}\cdot 36t^3 = 18t^3.$$

Für $t \neq 0$ ist das ungleich $0$, also gilt $r_2''(t) \not\perp r_2'(t)$.

---

**d)** Für die Parabel gilt:

$$r_3'(t) = \begin{pmatrix} 1 \\ 2t \end{pmatrix}, \qquad r_3''(t) = \begin{pmatrix} 0 \\ 2 \end{pmatrix}.$$

Die Geschwindigkeit ist

$$v_3(t) = \|r_3'(t)\| = \sqrt{1 + (2t)^2} = \sqrt{1 + 4t^2},$$

also nicht konstant. Das Skalarprodukt ist

$$\langle r_3'(t),\, r_3''(t)\rangle = \begin{pmatrix} 1 \\ 2t \end{pmatrix} \cdot \begin{pmatrix} 0 \\ 2 \end{pmatrix} = 4t.$$

Damit gilt $\langle r_3'(t),\, r_3''(t)\rangle = 0$ genau für $t = 0$.  
Für $t \neq 0$ sind Tangential- und Beschleunigungsvektor nicht orthogonal.
"""


def visualisierung_aufgabe_3_2(t):
    """
    Interaktive Visualisierung fuer Aufgabe 3.2.
    
    Drei parametrisierte Kurven werden verglichen:
      - r1 : Einheitskreis mit konstanter Geschwindigkeit
      - r2 : Derselbe Kreis, aber mit nicht-konstanter Geschwindigkeit (Umparametrisierung phi(t) = t^3)
      - r3 : Parabel
    
    Hauptaussage: Konstante Geschwindigkeit <=> r'(t) senkrecht r''(t) fuer alle t
    """

    fig = plt.figure(figsize=(17, 10))
    gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.4)

    t_bereich = np.linspace(-2, 2, 500)
    skala     = 0.25

    # ── r1 : Einheitskreis mit konstanter Geschwindigkeit ────────────────────
    def r1(s):        return np.array([np.cos(s), np.sin(s)])
    def r1_strich(s): return np.array([-np.sin(s), np.cos(s)])
    def r1_zweimal(s):return np.array([-np.cos(s), -np.sin(s)])
    def geschw1(s):   return np.linalg.norm(r1_strich(s))
    def skalar1(s):   return np.dot(r1_strich(s), r1_zweimal(s))

    # ── r2 : Derselbe Kreis, reparametrisiert mit phi(t) = t^3 ──────────────
    def r2(s):        return np.array([np.cos(s**3), np.sin(s**3)])
    def r2_strich(s): return 3*s**2 * np.array([-np.sin(s**3), np.cos(s**3)])
    def r2_zweimal(s):return (
        6*s    * np.array([-np.sin(s**3),  np.cos(s**3)]) +
        9*s**4 * np.array([-np.cos(s**3), -np.sin(s**3)])
    )
    def geschw2(s):   return np.linalg.norm(r2_strich(s))
    def skalar2(s):   return np.dot(r2_strich(s), r2_zweimal(s))

    # ── r3 : Parabel r3(t) = (t, t^2) ────────────────────────────────────────
    def r3(s):        return np.array([s, s**2])
    def r3_strich(s): return np.array([1.0, 2*s])
    def r3_zweimal(s):return np.array([0.0, 2.0])
    def geschw3(s):   return np.linalg.norm(r3_strich(s))
    def skalar3(s):   return np.dot(r3_strich(s), r3_zweimal(s))

    kurven = [
        (r1, r1_strich, r1_zweimal, geschw1, skalar1,
         r"$r_1$ -- Kreis (konst. Geschw.)", "green"),
        (r2, r2_strich, r2_zweimal, geschw2, skalar2,
         r"$r_2$ -- Kreis (umparametrisiert)", "orange"),
        (r3, r3_strich, r3_zweimal, geschw3, skalar3,
         r"$r_3$ -- Parabel", "purple"),
    ]

    for i, (r, rp, rs, geschw, skalar, titel, farbe) in enumerate(kurven):

        # ── Subplot oben : Kurve mit Vektoren ─────────────────────────────────
        ax_kurve = fig.add_subplot(gs[0, i])

        punkte        = np.array([r(s) for s in t_bereich])
        geschw_werte  = np.array([geschw(s) for s in t_bereich])
        v_max         = geschw_werte.max() + 1e-9

        for j in range(len(t_bereich) - 1):
            farbe_j = plt.cm.coolwarm(geschw_werte[j] / v_max)
            ax_kurve.plot(punkte[j:j+2, 0], punkte[j:j+2, 1],
                          color=farbe_j, lw=2)

        pos = r(t)
        vel = rp(t)
        acc = rs(t)
        d   = skalar(t)

        ax_kurve.plot(*pos, 'ko', ms=7, zorder=5)

        # Geschwindigkeitsvektor r'(t) in Blau
        norm_vel = np.linalg.norm(vel)
        if norm_vel > 1e-9:
            ax_kurve.annotate(
                "", xy=pos + skala * vel / norm_vel, xytext=pos,
                arrowprops=dict(arrowstyle="->", color="blue", lw=2.2)
            )

        # Beschleunigungsvektor r''(t) in Rot
        norm_acc = np.linalg.norm(acc)
        if norm_acc > 1e-9:
            ax_kurve.annotate(
                "", xy=pos + skala * acc / norm_acc, xytext=pos,
                arrowprops=dict(arrowstyle="->", color="red", lw=2.2)
            )

        # ── Orthogonalitaetsstatus ─────────────────────────────────────────────
        if norm_vel > 1e-8 and norm_acc > 1e-8:
            cos_winkel = np.clip(d / (norm_vel * norm_acc), -1, 1)
            winkel     = np.degrees(np.arccos(np.abs(cos_winkel)))
            if abs(d) < 0.05:
                status       = r"[OK] $r' \perp r''$"
                status_farbe = "green"
            else:
                status       = f"[X] Winkel: {winkel:.1f} Grad"
                status_farbe = "red"
        else:
            status       = "[?] Nullvektor"
            status_farbe = "gray"

        ax_kurve.set_title(
            f"{titel}\n"
            f"$|r'(t)| = {geschw(t):.2f}$ | "
            f"$\\langle r', r''\\rangle = {d:.2f}$ | {status}",
            fontsize=8.5,
            color=status_farbe
        )
        ax_kurve.set_aspect('equal')
        ax_kurve.grid(True, alpha=0.3)
        ax_kurve.set_xlabel("$x$", fontsize=9)
        ax_kurve.set_ylabel("$y$", fontsize=9)

        legende = [
            Line2D([0],[0], color='blue', lw=2, label="$r'(t)$ -- Geschwindigkeit"),
            Line2D([0],[0], color='red',  lw=2, label="$r''(t)$ -- Beschleunigung"),
        ]
        ax_kurve.legend(handles=legende, fontsize=7, loc='upper right')

        # ── Subplot unten : Skalarprodukt ──────────────────────────────────────
        ax_skal = fig.add_subplot(gs[1, i])

        t_plot        = np.linspace(-2, 2, 400)
        skalar_werte  = np.array([skalar(s) for s in t_plot])

        ax_skal.plot(t_plot, skalar_werte, color=farbe, lw=2,
                     label=r"$\langle r'(t), r''(t)\rangle$")
        ax_skal.axhline(0, color='gray',  lw=1,   ls='--', alpha=0.7)
        ax_skal.axvline(t, color='black', lw=1.5, ls='--', alpha=0.8,
                        label=f"$t = {t:.2f}$")
        ax_skal.plot(t, skalar(t), 'ko', ms=6, zorder=5)

        ax_skal.fill_between(
            t_plot, skalar_werte, 0,
            where=(np.abs(skalar_werte) < 0.3),
            alpha=0.25, color='green', label='$\\approx 0$ (orthogonal)'
        )

        ax_skal.set_xlabel("$t$", fontsize=9)
        ax_skal.set_ylabel(r"$\langle r'(t),\, r''(t)\rangle$", fontsize=9)
        ax_skal.set_title(f"Skalarprodukt -- {titel}", fontsize=8.5)
        ax_skal.grid(True, alpha=0.3)
        ax_skal.legend(fontsize=7, loc='upper right')

        y_spanne = max(abs(skalar_werte.min()), abs(skalar_werte.max()), 0.5)
        ax_skal.set_ylim(-y_spanne * 1.3, y_spanne * 1.3)

    # ── Farbleiste ────────────────────────────────────────────────────────────
    sm = plt.cm.ScalarMappable(cmap='coolwarm')
    sm.set_array([])
    cbar = fig.colorbar(
        sm, ax=fig.axes, orientation='horizontal',
        fraction=0.015, pad=0.02, shrink=0.35
    )
    cbar.set_label(
        "Geschwindigkeit $|r'(t)|$  (blau = langsam,  rot = schnell)",
        fontsize=9
    )

    fig.suptitle(
        f"Aufgabe 3.2 -- Konstante Geschwindigkeit $\\Leftrightarrow$ "
        f"$r'(t) \\perp r''(t)$  |  $t = {t:.2f}$",
        fontsize=13, fontweight='bold', y=1.02
    )
    plt.show()


# ── Aufruf ────────────────────────────────────────────────────────────────────
display(create_solution_toggle(
    problem_3_2,
    solution_3_2,
    "Loesung anzeigen",
    lambda: interact(
        visualisierung_aufgabe_3_2,
        t=FloatSlider(
            value=0.5,
            min=-1.8,
            max=1.8,
            step=0.05,
            description="Parameter $t$:",
            continuous_update=False,
            style={'description_width': 'initial'},
            layout={'width': '500px'}
        )
    )
))



In [9]:
problem_3_3 = r"""
<a id="aufgabe3"></a>
#### Aufgabe 3.3 — Tangente, Normale und Krümmung einer Kurve

**a)** Gegeben sei die Kurve

$$\mathbf{r}(t) = \begin{pmatrix} t \\ t^2 \end{pmatrix}, \quad t \in \mathbb{R}.$$

Bestimmen Sie die Tangente an die Kurve im Punkt $t_0 = 1$ in Parameterform
und in der Form $y = mx + b$.

**b)** Bestimmen Sie einen Normalenvektor $\mathbf{n}(t_0)$ im Punkt $t_0 = 1$
und geben Sie die Normale im Punkt $\mathbf{r}(1)$ als Geradengleichung an.

**c)** Bestimmen Sie die Krümmung $\kappa(t)$ der Kurve $\mathbf{r}(t)$.
Berechnen Sie $\kappa(1)$.

*Hinweis:* Für ebene Kurven gilt

$$\kappa(t) = \frac{|x'(t)\,y''(t) - y'(t)\,x''(t)|}{\bigl(x'(t)^2 + y'(t)^2\bigr)^{3/2}}.$$

**Versuchen Sie erstmal allein**     
<a href="#tipps">💡Lösungstipps</a>
"""

solution_3_3 = r"""
#### Lösung 3.3

---

**a)** Es gilt

$$\mathbf{r}'(t) = \begin{pmatrix} 1 \\ 2t \end{pmatrix}.$$

Im Punkt $t_0 = 1$ ist

$$\mathbf{r}(1) = \begin{pmatrix} 1 \\ 1 \end{pmatrix}, \qquad
  \mathbf{r}'(1) = \begin{pmatrix} 1 \\ 2 \end{pmatrix}.$$

Tangente in Parameterform:

$$\mathbf{T}(s) = \mathbf{r}(1) + s\,\mathbf{r}'(1)
               = \begin{pmatrix} 1 \\ 1 \end{pmatrix}
               + s \begin{pmatrix} 1 \\ 2 \end{pmatrix}.$$

Als Funktionsgleichung: Steigung $m = \dfrac{2}{1} = 2$, also

$$y - 1 = 2(x-1) \quad\Rightarrow\quad \boxed{y = 2x - 1.}$$

---

**b)** Ein Normalenvektor ist z.B.

$$\mathbf{n}(1) = \begin{pmatrix} -2 \\ 1 \end{pmatrix},$$

denn $\langle(1,2)^\top,(-2,1)^\top\rangle = -2 + 2 = 0$.
Normale durch $(1,1)$ in Parameterform:

$$\mathbf{N}(s) = \begin{pmatrix} 1 \\ 1 \end{pmatrix}
               + s \begin{pmatrix} -2 \\ 1 \end{pmatrix}.$$

Als Funktionsgleichung: Normalensteigung $-\tfrac{1}{2}$, also

$$y - 1 = -\tfrac{1}{2}(x-1)
\quad\Rightarrow\quad
\boxed{y = -\tfrac{1}{2}\,x + \tfrac{3}{2}.}$$

---

**c)** Für $x(t)=t,\; y(t)=t^2$ gilt:

$$x'(t)=1,\quad x''(t)=0,\quad y'(t)=2t,\quad y''(t)=2.$$

Damit

$$\kappa(t)
  = \frac{|x'(t)\,y''(t) - y'(t)\,x''(t)|}
         {(x'(t)^2 + y'(t)^2)^{3/2}}
  = \frac{|1\cdot 2 - 2t\cdot 0|}
         {(1 + 4t^2)^{3/2}}
  = \frac{2}{(1+4t^2)^{3/2}}.$$

Also

$$\kappa(1)
  = \frac{2}{(1+4)^{3/2}}
  = \frac{2}{5^{3/2}}
  = \boxed{\dfrac{2}{5\sqrt{5}}.}$$
"""


import matplotlib.patches as mpatches


def plot_kurve_3_3(t0=1.0):
    """
    Interaktive Visualisierung der Kurve r(t) = (t, t²)
    Zeigt Tangente, Normale und Krümmung am Punkt t0
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # ── Kurvenpunkte berechnen ──────────────────────────────
    t = np.linspace(-2.5, 2.5, 400)
    x_kurve = t
    y_kurve = t**2

    # Aktueller Punkt auf der Kurve
    x0 = t0
    y0 = t0**2

    # Tangentenvektor: r'(t0) = (1, 2t0), normiert
    tx, ty = 1.0, 2 * t0
    norm_t = np.sqrt(tx**2 + ty**2)
    tx_u, ty_u = tx / norm_t, ty / norm_t

    # Normalenvektor: senkrecht zur Tangente, normiert
    nx, ny = -2 * t0, 1.0
    norm_n = np.sqrt(nx**2 + ny**2)
    nx_u, ny_u = nx / norm_n, ny / norm_n

    # Länge der Pfeile
    pfeil_len = 1.2

    # Krümmung κ(t0) = 2 / (1 + 4t0²)^(3/2)
    kappa     = 2 / (1 + 4 * t0**2) ** 1.5
    kappa_all = 2 / (1 + 4 * t**2)  ** 1.5

    # ── LINKES BILD : Kurve + Tangente + Normale ───────────
    ax = axes[0]

    # Kurve zeichnen
    ax.plot(x_kurve, y_kurve, 'royalblue', lw=2.5,
            label=r'$\mathbf{r}(t)=(t,\,t^2)$')

    # Aktueller Punkt
    ax.plot(x0, y0, 'o', color='red', markersize=10, zorder=5,
            label=f'$r(t_0)=({x0:.2f},\,{y0:.2f})$')

    # Tangentenpfeil (beide Richtungen)
    ax.annotate("",
        xy=(x0 + pfeil_len * tx_u, y0 + pfeil_len * ty_u),
        xytext=(x0, y0),
        arrowprops=dict(arrowstyle="-|>", color="green", lw=2.5, mutation_scale=20))
    ax.annotate("",
        xy=(x0 - pfeil_len * tx_u, y0 - pfeil_len * ty_u),
        xytext=(x0, y0),
        arrowprops=dict(arrowstyle="-|>", color="green", lw=2.5, mutation_scale=20))

    # Normalenpfeil (beide Richtungen)
    ax.annotate("",
        xy=(x0 + pfeil_len * nx_u, y0 + pfeil_len * ny_u),
        xytext=(x0, y0),
        arrowprops=dict(arrowstyle="-|>", color="darkorange", lw=2.5, mutation_scale=20))
    ax.annotate("",
        xy=(x0 - pfeil_len * nx_u, y0 - pfeil_len * ny_u),
        xytext=(x0, y0),
        arrowprops=dict(arrowstyle="-|>", color="darkorange", lw=2.5, mutation_scale=20))

    # Beschriftung der Pfeile
    ax.text(x0 + pfeil_len * tx_u + 0.1, y0 + pfeil_len * ty_u,
            r"$\mathbf{T}$", color="green", fontsize=14, fontweight='bold')
    ax.text(x0 + pfeil_len * nx_u + 0.1, y0 + pfeil_len * ny_u,
            r"$\mathbf{N}$", color="darkorange", fontsize=14, fontweight='bold')

    # Legende zusammenbauen
    t_patch = mpatches.Patch(color='green',      label='Tangentenvektor')
    n_patch = mpatches.Patch(color='darkorange',  label='Normalenvektor')
    ax.legend(handles=[
        plt.Line2D([0],[0], color='royalblue', lw=2.5, label=r'$\mathbf{r}(t)=(t,t^2)$'),
        plt.Line2D([0],[0], color='red', marker='o', lw=0,
                   markersize=9, label=f'$t_0={t0:.1f}$'),
        t_patch, n_patch
    ], fontsize=11)

    # Achsen und Gitter
    ax.set_xlim(-2.8, 2.8)
    ax.set_ylim(-0.5, 7)
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.set_title(r'Kurve, Tangente & Normale bei $t_0$', fontsize=13, fontweight='bold')
    ax.axhline(0, color='gray', lw=0.8)
    ax.axvline(0, color='gray', lw=0.8)
    ax.grid(True, alpha=0.3)

    # Informationsbox mit Geradengleichungen
    steigung_t   =  2 * t0
    steigung_n   = -1 / (2 * t0) if t0 != 0 else float('inf')
    achsenabst_t = y0 - steigung_t * x0
    achsenabst_n = y0 - steigung_n * x0
    info = (f"$t_0 = {t0:.2f}$\n"
            f"$r(t_0) = ({x0:.2f},\\ {y0:.2f})$\n"
            f"Tangente: $y = {steigung_t:.2f}x {achsenabst_t:+.2f}$\n"
            f"Normale:  $y = {steigung_n:.2f}x {achsenabst_n:+.2f}$\n"
            f"Krümmung: $\\kappa = {kappa:.4f}$")
    ax.text(0.02, 0.97, info, transform=ax.transAxes,
            fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow',
                      edgecolor='gray', alpha=0.9))

    # ── RECHTES BILD : Krümmungsverlauf κ(t) ───────────────
    ax2 = axes[1]

    # Krümmungskurve
    ax2.plot(t, kappa_all, 'purple', lw=2.5,
             label=r'$\kappa(t) = \frac{2}{(1+4t^2)^{3/2}}$')

    # Aktueller t0-Wert markieren
    ax2.axvline(t0, color='red', lw=1.5, linestyle='--', label=f'$t_0={t0:.1f}$')
    ax2.plot(t0, kappa, 'o', color='red', markersize=10, zorder=5)
    ax2.text(t0 + 0.08, kappa + 0.01,
             f'$\\kappa({t0:.1f}) = {kappa:.4f}$',
             color='red', fontsize=11)

    # Achsen und Gitter
    ax2.set_xlabel('t', fontsize=12)
    ax2.set_ylabel(r'$\kappa(t)$', fontsize=12)
    ax2.set_title(r'Krümmungsverlauf $\kappa(t)$', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(-2.5, 2.5)
    ax2.set_ylim(0, 2.2)

    # Haupttitel
    plt.suptitle(
        r'Aufgabe 3.3 — Tangente, Normale und Krümmung von $\mathbf{r}(t)=(t,t^2)$',
        fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# ── Aufruf: Visualisierung hinter "Lösung anzeigen" verstecken ─────────────
display(create_solution_toggle(
    problem_3_3,
    solution_3_3,
    "Lösung anzeigen",
    lambda: interact(
        plot_kurve_3_3,
        t0=FloatSlider(
            value=1.0, min=-2.4, max=2.4, step=0.1,
            description='$t_0$',
            continuous_update=True,
            style={'description_width': 'initial'}
        )
    )
))




In [10]:
problem_3_4 = r"""
<a id="aufgabe4"></a>
#### Aufgabe 3.4 — Zwei Raketen: Kollisionskurs durch Wahl eines Parameters

Zwei Raketen werden als Punktmassen in der Ebene modelliert. Ihre Positionen (in
Metern) sind für $t \geq 0$ gegeben durch

$$\mathbf{r}_1(t) = \begin{pmatrix} 2t \\ 4-t \end{pmatrix}, \qquad
  \mathbf{r}_2(t) = \begin{pmatrix} \alpha+t \\ t \end{pmatrix},$$

wobei $\alpha \in \mathbb{R}$ ein wählbarer Parameter ist.

**a)** Skizzieren Sie beide Bahnen im $xy$-Koordinatensystem (für ein beliebiges $\alpha$)
und beschreiben Sie kurz, wie sich $\alpha$ geometrisch auswirkt.

**b)** Bestimmen Sie $\alpha$ so, dass sich die beiden Raketen zu einem Zeitpunkt $t \geq 0$
treffen, d.h.

$$\mathbf{r}_1(t) = \mathbf{r}_2(t) \quad \text{für ein } t \geq 0.$$

**c)** Bestimmen Sie den Treffzeitpunkt $t$ und den Treffpunkt $P$.

**Versuchen Sie erstmal allein**     
<a href="#tipps">💡Lösungstipps</a>
"""

solution_3_4 = r"""
#### Lösung 3.4

---

**a)** Die Bahn von $\mathbf{r}_1$ ist eine Gerade: $x_1(t) = 2t$ wächst, $y_1(t) = 4-t$ fällt.
Die Bahn von $\mathbf{r}_2$ ist ebenfalls eine Gerade: $y_2(t) = t$ wächst, und
$x_2(t) = \alpha + t$ ist gegenüber $\alpha$ in $x$-Richtung verschoben.
$\alpha$ verschiebt also die gesamte Bahn der zweiten Rakete parallel zur $x$-Achse.

---

**b)** Bedingung für ein Treffen:

$$\begin{pmatrix} 2t \\ 4-t \end{pmatrix} = \begin{pmatrix} \alpha+t \\ t \end{pmatrix}.$$

Dies entspricht dem Gleichungssystem

$$2t = \alpha + t, \qquad 4 - t = t.$$

Aus der zweiten Gleichung folgt

$$4 = 2t \iff t = 2.$$

In die erste Gleichung eingesetzt:

$$2 \cdot 2 = \alpha + 2 \iff 4 = \alpha + 2 \iff \boxed{\alpha = 2.}$$

---

**c)** Treffzeitpunkt ist $t = 2$. Der Treffpunkt ist

$$P = \mathbf{r}_1(2) = \begin{pmatrix} 2 \cdot 2 \\ 4-2 \end{pmatrix} = \boxed{\begin{pmatrix} 4 \\ 2 \end{pmatrix}.}$$
"""

def plot_raketen_3_4(alpha=2.0):
    t = np.linspace(0, 4, 300)

    x1, y1 = 2*t, 4 - t
    x2, y2 = alpha + t, t

    meets = np.isclose(alpha, 2.0, atol=0.05)

    fig, ax = plt.subplots(figsize=(7, 6))

    ax.plot(x1, y1, 'b-', lw=2, label=r'$\mathbf{r}_1(t) = (2t,\ 4-t)$')
    ax.plot(x2, y2, 'r-', lw=2, label=rf'$\mathbf{{r}}_2(t) = ({alpha:.1f}+t,\ t)$')

    # Startpunkte
    ax.plot(0, 4, 'bs', ms=8)
    ax.plot(alpha, 0, 'rs', ms=8)

    if meets:
        # Kollision bei t=2
        ax.plot(4, 2, 'ko', ms=10, zorder=5, label=r'Treffpunkt $P=(4,2)$')
        ax.annotate(r'$P=(4,2)$', xy=(4, 2), xytext=(4.3, 2.4), fontsize=12,
                    arrowprops=dict(arrowstyle='->', color='black'))
        ax.set_title(rf'$\alpha = {alpha:.1f}$ — Kollision bei $t=2$, $P=(4,2)$',
                     fontsize=13, color='green')

    else:
        # --- Geometrischer Schnittpunkt ---
        # r1: y = 4 - x/2
        # r2: y = x - alpha
        # => 4 - x/2 = x - alpha => x = (8 + 2*alpha) / 3
        x_cross = (8 + 2*alpha) / 3
        y_cross = x_cross - alpha

        t1_cross = x_cross / 2          # aus x1 = 2*t1
        t2_cross = x_cross - alpha      # aus x2 = alpha + t2

        r1_cross = (2*t1_cross, 4 - t1_cross)
        r2_cross = (alpha + t2_cross, t2_cross)

        # Punkte auf den Kurven
        ax.plot(r1_cross[0], r1_cross[1], 'bo', ms=9, zorder=6)
        ax.plot(r2_cross[0], r2_cross[1], 'ro', ms=9, zorder=6)

        # Kreuz am geometrischen Schnittpunkt
        ax.plot(x_cross, y_cross, 'X', color='purple', ms=12, zorder=5,
                label='Geometrischer Schnittpunkt')

        # Annotation r1
        ax.annotate(
            f'$\\mathbf{{r}}_1(t_1)$\n$t_1={t1_cross:.2f}$\n'
            f'$=({r1_cross[0]:.2f},\\,{r1_cross[1]:.2f})$',
            xy=(r1_cross[0], r1_cross[1]),
            xytext=(r1_cross[0] - 2.8, r1_cross[1] + 0.7),
            fontsize=9, color='blue',
            arrowprops=dict(arrowstyle='->', color='blue'))

        # Annotation r2
        ax.annotate(
            f'$\\mathbf{{r}}_2(t_2)$\n$t_2={t2_cross:.2f}$\n'
            f'$=({r2_cross[0]:.2f},\\,{r2_cross[1]:.2f})$',
            xy=(r2_cross[0], r2_cross[1]),
            xytext=(r2_cross[0] + 0.4, r2_cross[1] - 1.3),
            fontsize=9, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

        # Infobox
        ax.text(0.02, 0.97,
                f'Kein Treffen ($\\alpha = {alpha:.1f}$)\n'
                f'Gleicher Ort nur wenn:\n'
                f'$t_1 = {t1_cross:.2f} \\neq t_2 = {t2_cross:.2f}$',
                transform=ax.transAxes,
                fontsize=10, color='purple', verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='lavender', edgecolor='purple'))

        ax.set_title(rf'$\alpha = {alpha:.1f}$ — Kein Treffen', fontsize=13, color='gray')

    ax.set_xlabel('$x$', fontsize=13)
    ax.set_ylabel('$y$', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-1, 10)
    ax.set_ylim(-1, 6)
    ax.axhline(0, color='k', lw=0.8)
    ax.axvline(0, color='k', lw=0.8)
    plt.tight_layout()
    plt.show()

# ── Aufruf ─────────────────────────────────────────────────────────────────
display(create_solution_toggle(
    problem_3_4,
    solution_3_4,
    "Lösung anzeigen",
    lambda: interact(
        plot_raketen_3_4,
        alpha=FloatSlider(
            value=2.0,
            min=-2.0,
            max=6.0,
            step=0.1,
            description=r'$\alpha$',
            continuous_update=True,
            style={'description_width': 'initial'},
            layout={'width': '500px'}
        )
    )
))


In [11]:
problem_3_5 = r"""
<a id="aufgabe5"></a>
**Aufgabe 3.5 — Zwei Raketen auf Parabelbahnen**

Zwei Raketen werden als Punktmassen in der Ebene modelliert. Der Boden ist durch $y = 0$ beschrieben. Beide Raketen starten zum Zeitpunkt $t = 0$.

Die erste Rakete fliegt auf der Parabel

$$\mathbf{r}_1(t) = \begin{pmatrix} x_1(t) \\ y_1(t) \end{pmatrix} = \begin{pmatrix} t \\ -t^2 + 2t \end{pmatrix}, \qquad 0 \leq t \leq 2,$$

also startet sie bei $(0, 0)$ und landet bei $(2, 0)$. Die zweite Rakete startet bei $x = 5$ am Boden:

$$\mathbf{r}_2(t) = \begin{pmatrix} x_2(t) \\ y_2(t) \end{pmatrix} = \begin{pmatrix} 5 - 2t \\ -2t^2 + \frac{11}{3}t \end{pmatrix}, \qquad t \geq 0.$$

(Beachten Sie: Die zweite Rakete muss nicht zur selben Zeit wie die erste Rakete wieder den Boden erreichen.)

- **a)** Bestimmen Sie die Startpunkte beider Raketen.
- **b)** Bestimmen Sie die Zeitpunkte, zu denen die Raketen den Boden berühren (also $y_1(t) = 0$ bzw. $y_2(t) = 0$ für $t > 0$).
- **c)** Fliegen beide Raketen in positiver $x$-Richtung? Begründen Sie Ihre Antwort.
- **d)** Bestimmen Sie, ob es einen Zeitpunkt $t > 0$ gibt, zu dem sich die beiden Raketen treffen. Falls ja: Bestimmen Sie den Treffzeitpunkt und den Treffpunkt $P$.
- **e)** Liegt der Treffpunkt über dem Boden? Begründen Sie kurz.

**Versuchen Sie erstmal allein**     
<a href="#tipps">💡Lösungstipps</a>
"""

solution_3_5 = r"""
**Lösung 3.5**

**a)**

$$\mathbf{r}_1(0) = \begin{pmatrix} 0 \\ 0 \end{pmatrix}, \qquad \mathbf{r}_2(0) = \begin{pmatrix} 5 \\ 0 \end{pmatrix}.$$

**b)** Für Rakete 1:

$$y_1(t) = -t^2 + 2t = t(2 - t) = 0 \implies t = 0 \text{ oder } t = 2.$$

Also berührt sie den Boden (nach dem Start) bei $t = 2$.

Für Rakete 2:

$$y_2(t) = -2t^2 + \frac{11}{3}t = t\!\left(-2t + \frac{11}{3}\right) = 0 \implies t = 0 \text{ oder } t = \frac{11}{6}.$$

Also berührt sie den Boden (nach dem Start) bei $t = \dfrac{11}{6}$.

**c)**

$$x_1'(t) = 1 > 0 \implies \text{Rakete 1 fliegt nach rechts},$$

$$x_2'(t) = -2 < 0 \implies \text{Rakete 2 fliegt nach links}.$$

**d)** Ein Treffen erfordert $x_1(t) = x_2(t)$ und $y_1(t) = y_2(t)$. Aus den $x$-Komponenten:

$$t = 5 - 2t \iff 3t = 5 \iff t = \frac{5}{3}.$$

Nun prüfen wir die $y$-Komponenten bei $t = \dfrac{5}{3}$:

$$y_1\!\left(\frac{5}{3}\right) = -\left(\frac{5}{3}\right)^2 + 2 \cdot \frac{5}{3} = -\frac{25}{9} + \frac{10}{3} = -\frac{25}{9} + \frac{30}{9} = \frac{5}{9},$$

$$y_2\!\left(\frac{5}{3}\right) = -2\left(\frac{5}{3}\right)^2 + \frac{11}{3} \cdot \frac{5}{3} = -\frac{50}{9} + \frac{55}{9} = \frac{5}{9}.$$

Damit treffen sich die Raketen bei

$$t = \frac{5}{3} \quad \text{im Punkt} \quad P = \mathbf{r}_1\!\left(\frac{5}{3}\right) = \begin{pmatrix} \frac{5}{3} \\ \frac{5}{9} \end{pmatrix}.$$

Außerdem liegt $\dfrac{5}{3} < 2$ und $\dfrac{5}{3} < \dfrac{11}{6}$, d.h. das Treffen findet statt, bevor beide Raketen gelandet sind.

**e)** Da $y(P) = \dfrac{5}{9} > 0$, liegt der Treffpunkt über dem Boden.
"""

import matplotlib.patches as mpatches

def plot_raketen_3_5():
    fig, ax = plt.subplots(figsize=(10, 5))

    # Trajectoires
    t1 = np.linspace(0, 2, 300)
    x1 = t1
    y1 = -t1**2 + 2*t1

    t2 = np.linspace(0, 11/6, 300)
    x2 = 5 - 2*t2
    y2 = -2*t2**2 + (11/3)*t2

    # Point de collision
    t_coll = 5/3
    xp = t_coll
    yp = -(t_coll)**2 + 2*t_coll  # = 5/9

    # Sol
    ax.axhline(0, color='black', linewidth=2)

    # Tracé des trajectoires
    ax.plot(x1, y1, color='blue', linewidth=2.5, label=r'$\mathbf{r}_1$')
    ax.plot(x2, y2, color='red', linewidth=2.5, label=r'$\mathbf{r}_2$')

    # Flèches sur les trajectoires
    ax.annotate("", xy=(x1[180], y1[180]), xytext=(x1[170], y1[170]),
                arrowprops=dict(arrowstyle="->", color='blue', lw=2))
    ax.annotate("", xy=(x2[180], y2[180]), xytext=(x2[170], y2[170]),
                arrowprops=dict(arrowstyle="->", color='red', lw=2))

    # Points remarquables sur le sol
    points_sol = {
        r'$\mathbf{r}_1(0)$': (0, 0),
        r'$\mathbf{r}_2(0)$': (5, 0),
        r'$\mathbf{r}_1(2)$': (2, 0),
        r'$\mathbf{r}_2\!\left(\frac{11}{6}\right)$': (5 - 2*(11/6), 0),
    }

    for label, (px, py) in points_sol.items():
        ax.plot(px, py, 'ko', markersize=5)
        ax.annotate(label, xy=(px, py), xytext=(0, -22),
                    textcoords='offset points', ha='center', fontsize=9)

    # Point de collision
    ax.plot(xp, yp, 'go', markersize=8, zorder=5)
    ax.annotate(
        r'$P\!\left(\dfrac{5}{3},\,\dfrac{5}{9}\right)$',
        xy=(xp, yp),
        xytext=(xp + 0.15, yp + 0.08),
        fontsize=10,
        color='darkgreen'
    )

    # Labels r1 r2 sur les courbes
    ax.text(0.15, 0.25, r'$\mathbf{r}_1$', color='blue', fontsize=12, fontweight='bold')
    ax.text(4.5, 0.25, r'$\mathbf{r}_2$', color='red', fontsize=12, fontweight='bold')

    # Label sol
    ax.text(5.2, -0.05, r'Boden $y = 0$', fontsize=9, va='top')

    # Axes
    ax.set_xlim(-0.5, 6)
    ax.set_ylim(-0.3, 1.4)
    ax.set_xlabel(r'$x$', fontsize=12)
    ax.set_ylabel(r'$y$', fontsize=12)
    ax.spines['left'].set_position('zero')
    ax.spines['bottom'].set_position('zero')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.set_title("Skizze: Zwei Raketen auf Parabelbahnen", fontsize=12, pad=15)

    plt.tight_layout()
    plt.show()


display(create_solution_toggle(
    problem_3_5,
    solution_3_5,
    "Lösung anzeigen",
    lambda: plot_raketen_3_5()
))


In [12]:
problem_3_6 = r"""
<a id="aufgabe6"></a>
**Aufgabe 3.6: Parameter eliminieren: Kreis, Ellipse, Parabel, Hyperbel**

Eine parametrisierte Kurve in der Ebene ist eine Abbildung

$$r : I \to \mathbb{R}^2, \qquad t \mapsto r(t) = \begin{pmatrix} x(t) \\ y(t) \end{pmatrix},$$

wobei $I \subset \mathbb{R}$ ein Intervall ist. Der Parameter $t$ beschreibt die Bewegung entlang der Kurve, und die Spurkurve ist die Menge aller Punkte $\{r(t) \mid t \in I\}$. Oft möchte man den Parameter eliminieren und die Kurve als Gleichung in $x$ und $y$ darstellen.

Bestimmen Sie zu jeder parametrisierten Kurve eine kartesische Gleichung ohne Parameter (d.h. eliminieren Sie $t$). Geben Sie außerdem sinnvolle Wertebereiche für $x$ und/oder $y$ an und skizzieren Sie die Kurve.

a) Kreis (Radius 2):
$$x(t) = 2\cos t, \qquad y(t) = 2\sin t, \qquad 0 \leq t < 2\pi.$$

b) Ellipse (Halbachsen 1 und 2):
$$x(t) = \cos t, \qquad y(t) = 2\sin t, \qquad 0 \leq t < 2\pi.$$

c) Parabel:
$$x(t) = t, \qquad y(t) = t^2, \qquad t \in \mathbb{R}.$$

d) Hyperbel:
$$x(t) = \cosh t, \qquad y(t) = \sinh t, \qquad t \in \mathbb{R}.$$

Aufgaben zu jeder Kurve:

1. Eliminieren Sie $t$ und geben Sie eine Gleichung in $x$ und $y$ an.

2. Nennen Sie den Wertebereich von $x$ (und ggf. von $y$), der aufgrund der Parametrisierung möglich ist.

3. Skizzieren Sie die Kurve im $xy$-Koordinatensystem.


**Versuchen Sie erstmal allein**     
<a href="#tipps">💡Lösungstipps</a>
"""

solution_3_6 = r"""
**Lösung 3.6:**

**a) Kreis (Radius 2):**

Aus $x = 2\cos t$ und $y = 2\sin t$ folgt

$$\left(\frac{x}{2}\right)^2 + \left(\frac{y}{2}\right)^2 = \cos^2 t + \sin^2 t = 1 \implies x^2 + y^2 = 4.$$

Existenzbedingungen:
$$-2 \leq x \leq 2, \qquad -2 \leq y \leq 2.$$

---

**b) Ellipse (Halbachsen 1 und 2):**

Aus $x = \cos t$ und $y = 2\sin t$ folgt

$$x^2 + \left(\frac{y}{2}\right)^2 = \cos^2 t + \sin^2 t = 1 \implies x^2 + \frac{y^2}{4} = 1.$$

Existenzbedingungen:
$$-1 \leq x \leq 1, \qquad -2 \leq y \leq 2.$$

---

**c) Parabel:**

Aus $x = t$ folgt $t = x$. Einsetzen in $y = t^2$ ergibt

$$y = x^2.$$

Existenzbedingungen:
$$x \in \mathbb{R}, \qquad y \geq 0.$$

---

**d) Hyperbel:**

Es gilt $\cosh^2 t - \sinh^2 t = 1$. Mit $x = \cosh t$, $y = \sinh t$ folgt

$$x^2 - y^2 = 1.$$

Existenzbedingungen:
$$x \geq 1, \qquad y \in \mathbb{R},$$

(die Parametrisierung beschreibt nur den rechten Ast).

"""


def plot_kurven_3_6():
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # ── a) Kreis ────────────────────────────────────────────────────────────
    ax = axes[0, 0]
    t = np.linspace(0, 2*np.pi, 500)
    ax.plot(2*np.cos(t), 2*np.sin(t), 'b-', linewidth=2)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_xlabel('$x$', fontsize=12)
    ax.set_ylabel('$y$', fontsize=12)
    ax.set_title('a) Kreis', fontsize=13)
    ax.text(0.5, 2.6, r'$x^2 + y^2 = 4$', color='blue', fontsize=11)
    # Points remarquables
    points = [(2, 0, '$(2,0)$', 'right'), (-2, 0, '$(-2,0)$', 'left'),
              (0, 2, '$(0,2)$', 'center'), (0, -2, '$(0,-2)$', 'center')]
    for px, py, label, ha in points:
        ax.plot(px, py, 'ko', markersize=5)
        dy = -0.35 if py < 0 else 0.25
        ax.annotate(label, (px, py), (px, py + dy),
                    fontsize=9, ha=ha, color='darkred')

    # ── b) Ellipse ──────────────────────────────────────────────────────────
    ax = axes[0, 1]
    ax.plot(np.cos(t), 2*np.sin(t), 'r-', linewidth=2)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_xlabel('$x$', fontsize=12)
    ax.set_ylabel('$y$', fontsize=12)
    ax.set_title('b) Ellipse', fontsize=13)
    ax.text(-0.2, 2.7, r'$x^2 + \dfrac{y^2}{4} = 1$', color='red', fontsize=11)
    points_e = [(1, 0, '$(1,0)$', 'right'), (-1, 0, '$(-1,0)$', 'left'),
                (0, 2, '$(0,2)$', 'center'), (0, -2, '$(0,-2)$', 'center')]
    for px, py, label, ha in points_e:
        ax.plot(px, py, 'ko', markersize=5)
        dy = -0.35 if py < 0 else 0.25
        ax.annotate(label, (px, py), (px, py + dy),
                    fontsize=9, ha=ha, color='darkred')

    # ── c) Parabel ──────────────────────────────────────────────────────────
    ax = axes[1, 0]
    x = np.linspace(-2.5, 2.5, 500)
    ax.plot(x, x**2, 'b-', linewidth=2)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-0.5, 5)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_xlabel('$x$', fontsize=12)
    ax.set_ylabel('$y$', fontsize=12)
    ax.set_title('c) Parabel', fontsize=13)
    ax.text(1.2, 4.2, r'$y = x^2$', color='blue', fontsize=11)
    points_p = [(0, 0, '$(0,0)$'), (1, 1, '$(1,1)$'), (-1, 1, '$(-1,1)$')]
    offsets = [(0.15, -0.35), (0.2, 0.2), (-0.5, 0.2)]
    for (px, py, label), (dx, dy) in zip(points_p, offsets):
        ax.plot(px, py, 'ko', markersize=5)
        ax.annotate(label, (px, py), (px+dx, py+dy),
                    fontsize=9, color='darkred')

    # ── d) Hyperbel ─────────────────────────────────────────────────────────
    ax = axes[1, 1]
    t_h = np.linspace(-2.5, 2.5, 500)
    ax.plot(np.cosh(t_h), np.sinh(t_h), 'r-', linewidth=2)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim(-0.5, 7)
    ax.set_ylim(-6, 6)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.set_xlabel('$x$', fontsize=12)
    ax.set_ylabel('$y$', fontsize=12)
    ax.set_title('d) Hyperbel', fontsize=13)
    ax.text(2.5, 4.5, r'$x^2 - y^2 = 1$', color='red', fontsize=11)
    ax.text(1.5, -5, r'rechter Ast $(x \geq 1)$', color='gray', fontsize=9)
    ax.plot(1, 0, 'ko', markersize=5)
    ax.annotate('$(1,0)$', (1, 0), (1.2, -0.6), fontsize=9, color='darkred')

    plt.tight_layout(pad=3.0)
    plt.show()

# ── Aufruf ──────────────────────────────────────────────────────────────────
display(create_solution_toggle(
    problem_3_6,
    solution_3_6,
    "Lösung anzeigen",
    lambda: plot_kurven_3_6()
))


In [13]:
problem_3_7 = r"""
<a id="aufgabe7"></a>
**Aufgabe 3.7(Teil I): Tangente und Normalenvektor in $\mathbb{R}^2$**

a) Ebene Kurve ($\mathbb{R}^2$). Gegeben sei

$$r(t) = \begin{pmatrix} t^2 \\ t^3 - t \end{pmatrix}, \qquad t \in \mathbb{R}.$$

b) Bestimmen Sie den Punkt $P = r(1)$.

c) Bestimmen Sie den Tangentenvektor $r'(1)$ und geben Sie die Tangente im Punkt $P$ in Parameterform an.

d) Bestimmen Sie einen Normalenvektor $\boldsymbol{n}$ an die Kurve im Punkt $P$ (also $\boldsymbol{n} \perp r'(1)$) und geben Sie die Normale im Punkt $P$ in Parameterform an.

e) Eliminieren Sie bei der Tangente den Parameter und geben Sie die Geradengleichung der Tangente in der Form $y = y(x)$ an.

**Versuchen Sie erstmal allein**     
<a href="#tipps">💡Lösungstipps</a>
"""

solution_3_7 = r"""
**Lösung 3.7:**

**a)** $\mathbb{R}^2$.

**b)**

$$P = r(1) = \begin{pmatrix} 1^2 \\ 1^3 - 1 \end{pmatrix} = \begin{pmatrix} 1 \\ 0 \end{pmatrix}.$$

**c)**

$$r'(t) = \begin{pmatrix} 2t \\ 3t^2 - 1 \end{pmatrix} \implies r'(1) = \begin{pmatrix} 2 \\ 2 \end{pmatrix}.$$

Tangente in Parameterform:

$$\ell(s) = P + s\, r'(1) = \begin{pmatrix} 1 \\ 0 \end{pmatrix} + s \begin{pmatrix} 2 \\ 2 \end{pmatrix}, \qquad s \in \mathbb{R}.$$

**d)** Ein Normalenvektor $\boldsymbol{n}$ muss $\boldsymbol{n} \cdot (2, 2)^T = 0$ erfüllen. Z.B.

$$\boldsymbol{n} = \begin{pmatrix} 1 \\ -1 \end{pmatrix} \qquad \text{denn} \quad (1,-1)\cdot(2,2) = 2 - 2 = 0.$$

Normale:

$$m(s) = P + s\,\boldsymbol{n} = \begin{pmatrix} 1 \\ 0 \end{pmatrix} + s \begin{pmatrix} 1 \\ -1 \end{pmatrix}, \qquad s \in \mathbb{R}.$$

**e)** Für die Tangente gilt

$$x = 1 + 2s, \qquad y = 0 + 2s.$$

Aus $x = 1 + 2s$ folgt $s = \dfrac{x-1}{2}$. Einsetzen in $y = 2s$ ergibt

$$y = 2 \cdot \frac{x-1}{2} = x - 1.$$

Also lautet die Tangente in der Form $y = y(x)$:

$$y = x - 1.$$
"""


def plot_tangente_normale_3_7():
    """
    Interaktive Visualisierung der Tangente und Normale
    einer parametrischen Kurve r(t) = (t^2, t^3 - t).
    """
    
    def aktualisieren(t0):
        fig, ax = plt.subplots(figsize=(8, 8))
        
        # ── Kurve zeichnen ───────────────────────────────────────
        t = np.linspace(-2, 2, 500)
        x = t**2
        y = t**3 - t
        ax.plot(x, y, 'b-', linewidth=2,
                label=r'$r(t)=(t^2,\ t^3-t)$')
        
        # ── Punkt P berechnen und einzeichnen ────────────────────
        Px = t0**2
        Py = t0**3 - t0
        ax.plot(Px, Py, 'ko', markersize=8, zorder=5)
        ax.annotate(f'$P = ({Px:.2f},\\ {Py:.2f})$',
                    (Px, Py), (Px + 0.15, Py + 0.2),
                    fontsize=11, color='black')
        
        # ── Tangentenvektor r'(t0) berechnen ─────────────────────
        dx = 2 * t0          # Ableitung von t^2
        dy = 3 * t0**2 - 1   # Ableitung von t^3 - t
        skalierung = 0.5

        # Tangentenpfeil zeichnen
        ax.annotate('',
                    xy=(Px + skalierung*dx, Py + skalierung*dy),
                    xytext=(Px, Py),
                    arrowprops=dict(arrowstyle='->', color='green',
                                   lw=2.5))
        ax.text(Px + skalierung*dx + 0.1, Py + skalierung*dy,
                f"$r'(t_0)=({dx:.2f},\\ {dy:.2f})$",
                color='green', fontsize=11)
        
        # ── Normalenvektor berechnen (senkrecht zur Tangente) ────
        # n muss n · r'(t0) = 0 erfüllen => n = (-dy, dx)
        nx = -dy
        ny =  dx
        norm = np.sqrt(nx**2 + ny**2)

        # Normalenvektor normieren (Länge = 1)
        if norm > 1e-6:
            nx_u, ny_u = nx / norm, ny / norm
        else:
            nx_u, ny_u = 0, 0

        # Normalenpfeil zeichnen
        ax.annotate('',
                    xy=(Px + skalierung*nx_u, Py + skalierung*ny_u),
                    xytext=(Px, Py),
                    arrowprops=dict(arrowstyle='->', color='orange',
                                   lw=2.5))
        ax.text(Px + skalierung*nx_u + 0.1, Py + skalierung*ny_u,
                f'$n=({nx:.2f},\\ {ny:.2f})$',
                color='orange', fontsize=11)
        
        # ── Tangentengerade einzeichnen ──────────────────────────
        # Parameterform: l(s) = P + s * r'(t0)
        if abs(dx) > 1e-6:
            x_gerade = np.linspace(0, 4, 200)
            steigung = dy / dx   # Steigung der Tangente
            y_gerade = Py + steigung * (x_gerade - Px)
            ax.plot(x_gerade, y_gerade, 'g--', alpha=0.5,
                    linewidth=1.5,
                    label=f'Tangente (Steigung={steigung:.2f})')
        
        # ── Koordinatenachsen und Gitter ─────────────────────────
        ax.axhline(0, color='black', linewidth=0.8)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlim(-0.5, 4.5)
        ax.set_ylim(-3, 3)
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.set_xlabel('$x$', fontsize=13)
        ax.set_ylabel('$y$', fontsize=13)
        ax.set_title(
            f'Tangente und Normale im Punkt $t_0 = {t0:.2f}$',
            fontsize=14)
        ax.legend(fontsize=10)
        
        # ── Informationsfeld (oben links) ────────────────────────
        info_text = (
            f"$t_0 = {t0:.2f}$\n"
            f"$P = ({Px:.2f},\\ {Py:.2f})$\n"
            f"$r'(t_0) = ({dx:.2f},\\ {dy:.2f})$\n"
            f"$n = ({nx:.2f},\\ {ny:.2f})$"
        )
        ax.text(0.02, 0.97, info_text,
                transform=ax.transAxes,
                fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round',
                          facecolor='lightyellow',
                          alpha=0.8))
        
        plt.tight_layout()
        plt.show()
    
    # ── Schieberegler für t0 ─────────────────────────────────────
    schieberegler = widgets.FloatSlider(
        value=1.0,
        min=-2.0,
        max=2.0,
        step=0.05,
        description='$t_0$:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    
    # ── Beschriftung über dem Schieberegler ──────────────────────
    titel = widgets.HTML(
        value="<b>Verschieben Sie den Schieberegler, "
              "um den Punkt auf der Kurve zu bewegen:</b>"
    )
    
    # ── Alles zusammen anzeigen ──────────────────────────────────
    ausgang = widgets.interactive(aktualisieren, t0=schieberegler)
    display(widgets.VBox([titel, ausgang]))


# ════════════════════════════════════════════════════════════════
# Aufruf: Plot wird nur angezeigt, wenn der Student auf
# "Lösung anzeigen" klickt
# ════════════════════════════════════════════════════════════════
display(create_solution_toggle(
    problem_3_7,
    solution_3_7,
    "Lösung anzeigen",
    lambda: plot_tangente_normale_3_7()
))


In [14]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown
from ipywidgets import VBox, Button, Output
from matplotlib import cm

# ════════════════════════════════════════════════════════════════
# Umbenannte Funktion: create_solution_toggle_3D
# ════════════════════════════════════════════════════════════════

def create_solution_toggle_3D(problem_markdown, solution_markdown,
                               solution_title="Lösung anzeigen",
                               plot_callback=None):
    """
    Erstellt ein interaktives Toggle-Element für Lösungen mit optionalem 3D-Widget.

    Parameter:
        problem_markdown  : Aufgabentext im Markdown-Format (LaTeX unterstützt)
        solution_markdown : Lösungstext im Markdown-Format (LaTeX unterstützt)
        solution_title    : Beschriftung des Buttons
        plot_callback     : Optionale Funktion, die beim Anzeigen der Lösung aufgerufen wird
                            (z.B. für interaktive 3D-Darstellungen)
    """
    # Aufgaben-Container als Output-Widget
    problem_output = Output()
    with problem_output:
        display(Markdown(f"""
<div style="background-color: #f9f9f9; border-left: 4px solid #0066cc; 
            padding: 20px; margin: 20px 0; border-radius: 5px;">
<h3 style="color: #333;">📝</h3>

{problem_markdown}

</div>
        """))

    # Toggle-Button
    button = Button(
        description=f"▶ {solution_title}",
        button_style='info',
        tooltip='Klicken zum Anzeigen der Lösung',
        layout={'width': '250px', 'padding': '10px'}
    )

    # Ausgabe-Container für Lösung und Widget
    output   = Output()
    is_shown = [False]  # Zustandsverfolgung (sichtbar / verborgen)

    def toggle_solution(b):
        output.clear_output()
        is_shown[0] = not is_shown[0]

        if is_shown[0]:
            # Lösung einblenden
            button.description = "▼ Lösung verbergen"
            with output:
                display(Markdown(f"""
<div style="margin-top: 15px; background-color: #e8f4f8; 
            padding: 15px; border-radius: 5px; border: 1px solid #00ccff;">

{solution_markdown}

</div>
                """))
                # 3D-Widget anzeigen, falls ein Callback übergeben wurde
                if plot_callback is not None:
                    plot_callback()
        else:
            # Lösung ausblenden
            button.description = f"▶ {solution_title}"

    button.on_click(toggle_solution)
    return VBox([problem_output, button, output])


# ════════════════════════════════════════════════════════════════
# 3D-Zeichenfunktion: Raumkurve mit Frenet-Dreibein
# ════════════════════════════════════════════════════════════════

def zeichne_raumkurve_3d(t0, tangente_zeigen, normal_zeigen,
                          punkt_zeigen, schmiegekreis_zeigen):
    """
    Zeichnet die Raumkurve s(t) = (cos t, sin t, t) mit dem Frenet-Dreibein
    am Punkt P = s(t0) in einer interaktiven 3D-Darstellung.
    """
    fig = plt.figure(figsize=(11, 7))
    ax  = fig.add_subplot(111, projection='3d')

    # Raumkurve farbig darstellen
    t          = np.linspace(-2*np.pi, 2*np.pi, 600)
    x, y, z    = np.cos(t), np.sin(t), t
    norm_farbe  = plt.Normalize(z.min(), z.max())
    farben      = cm.plasma(norm_farbe(z))
    for i in range(len(t) - 1):
        ax.plot(x[i:i+2], y[i:i+2], z[i:i+2],
                color=farben[i], linewidth=2)

    # Punkt P und Ableitungen berechnen
    P      = np.array([np.cos(t0),  np.sin(t0), t0])
    T      = np.array([-np.sin(t0), np.cos(t0), 1.0])
    T_norm = T / np.linalg.norm(T)
    Tpp    = np.array([-np.cos(t0), -np.sin(t0), 0.0])
    N_roh  = Tpp - np.dot(Tpp, T_norm) * T_norm
    lng    = np.linalg.norm(N_roh)
    N_norm = N_roh / lng if lng > 1e-10 else N_roh
    B_norm = np.cross(T_norm, N_norm)

    # Skalierung der Vektoren
    sk = 1.0

    # Punkt P einzeichnen
    if punkt_zeigen:
        ax.scatter(*P, color='red', s=120, zorder=5,
                   label=f'P = s({t0:.2f})')
        ax.text(P[0]+0.05, P[1]+0.05, P[2]+0.1, 'P', color='red', fontsize=11)

    # Tangentenvektor T einzeichnen
    if tangente_zeigen:
        ax.quiver(*P, *(sk * T_norm),
                  color='blue', linewidth=2, arrow_length_ratio=0.15,
                  label='Tangente T̂')

    # Hauptnormalenvektor N einzeichnen
    if normal_zeigen:
        ax.quiver(*P, *(sk * N_norm),
                  color='green', linewidth=2, arrow_length_ratio=0.15,
                  label='Hauptnormale N')

    # Schmiegekreis einzeichnen
    if schmiegekreis_zeigen:
        # Krümmungsradius: kappa = 1/2 → R = 2
        kappa = 0.5
        R     = 1.0 / kappa
        # Mittelpunkt des Schmiegekreises
        M     = P + R * N_norm
        phi   = np.linspace(0, 2*np.pi, 200)
        # Kreis in der Schmiegeebene (T, N)
        kreis = np.array([
            M + R * (np.cos(ph) * N_norm + np.sin(ph) * T_norm) * (-1)
            for ph in phi
        ])
        ax.plot(kreis[:, 0], kreis[:, 1], kreis[:, 2],
                color='orange', linewidth=1.8, linestyle='--',
                label=f'Schmiegekreis (R={R:.1f})')
        ax.scatter(*M, color='orange', s=60, zorder=4)
        ax.text(M[0]+0.05, M[1]+0.05, M[2]+0.1, 'M', color='orange', fontsize=10)

    # Achsenbeschriftung und Legende
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.set_zlabel('z', fontsize=11)
    ax.set_title(f'Raumkurve s(t) = (cos t, sin t, t)  –  t₀ = {t0:.2f}',
                 fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)
    ax.view_init(elev=25, azim=45)
    plt.tight_layout()
    plt.show()


# ════════════════════════════════════════════════════════════════
# Interaktives Widget aufbauen
# ════════════════════════════════════════════════════════════════

def plot_callback_3_7_b():
    """
    Erstellt das interaktive 3D-Widget für Aufgabe 3.7b
    und zeigt es direkt an (wird vom plot_callback aufgerufen).
    """
    # Zeitparameter-Schieberegler
    schieber_t = widgets.FloatSlider(
        value=np.pi/2, min=-2*np.pi, max=2*np.pi, step=0.05,
        description='t₀',
        style={'description_width': '30px'},
        layout=widgets.Layout(width='500px'),
        readout_format='.2f'
    )

    # Toggle-Buttons für Anzeigeoptionen
    kn_tangente = widgets.ToggleButton(value=True,  description='Tangente T̂',
                                       button_style='primary',
                                       layout=widgets.Layout(width='130px'))
    kn_normal   = widgets.ToggleButton(value=True,  description='Normale N',
                                       button_style='success',
                                       layout=widgets.Layout(width='130px'))
    kn_punkt    = widgets.ToggleButton(value=True,  description='Punkt P',
                                       button_style='danger',
                                       layout=widgets.Layout(width='130px'))
    kn_schmieg  = widgets.ToggleButton(value=False, description='Schmiegekreis',
                                       button_style='warning',
                                       layout=widgets.Layout(width='130px'))

    # Schnellzugriff-Buttons für besondere Parameterwerte
    kn_null = widgets.Button(description='t = 0',
                              layout=widgets.Layout(width='100px'))
    kn_Q    = widgets.Button(description='t = π/2  (Q)',
                              layout=widgets.Layout(width='130px'))
    kn_pi   = widgets.Button(description='t = π',
                              layout=widgets.Layout(width='100px'))

    def gehe_null(b): schieber_t.value = 0.0
    def gehe_Q(b):    schieber_t.value = np.pi / 2
    def gehe_pi(b):   schieber_t.value = np.pi

    kn_null.on_click(gehe_null)
    kn_Q.on_click(gehe_Q)
    kn_pi.on_click(gehe_pi)

    # Interaktive Ausgabe verknüpfen
    ausgabe = widgets.interactive_output(
        zeichne_raumkurve_3d,
        dict(
            t0                   = schieber_t,
            tangente_zeigen      = kn_tangente,
            normal_zeigen        = kn_normal,
            punkt_zeigen         = kn_punkt,
            schmiegekreis_zeigen = kn_schmieg,
        )
    )

    # Gesamtes Widget zusammenstellen und anzeigen
    viz = VBox([
        widgets.HTML("<b>Interaktive 3D-Darstellung – Frenet-Dreibein</b>"),
        schieber_t,
        widgets.HBox([kn_tangente, kn_normal, kn_punkt, kn_schmieg]),
        widgets.HTML("<i>Schnellzugriff auf besondere Parameterwerte:</i>"),
        widgets.HBox([kn_null, kn_Q, kn_pi]),
        ausgabe,
    ])
    display(viz)


# ════════════════════════════════════════════════════════════════
# Aufgaben- und Lösungstext (Markdown + LaTeX)
# ════════════════════════════════════════════════════════════════

problem_3_7_b = r"""
### Aufgabe 3.7 – Raumkurve ($\mathbb{R}^3$)

Gegeben sei die Raumkurve:

$$s(t) = \begin{pmatrix} \cos t \\ \sin t \\ t \end{pmatrix}, \quad t \in \mathbb{R}$$

**f)** Es handelt sich um eine Raumkurve in $\mathbb{R}^3$.

**g)** Bestimmen Sie den Punkt $Q = s\!\left(\tfrac{\pi}{2}\right)$.

**h)** Bestimmen Sie den Tangentenvektor $s'\!\left(\tfrac{\pi}{2}\right)$ und geben Sie die Tangente im Punkt $Q$ in Parameterform an.

**i)** Bestimmen Sie einen Vektor $n \neq 0$, der im Punkt $Q$ **senkrecht** auf dem Tangentenvektor steht, also:

$$\left\langle n,\, s'\!\left(\tfrac{\pi}{2}\right) \right\rangle = 0$$

Geben Sie einen solchen Normalenvektor explizit an.  
*(Hinweis: In $\mathbb{R}^3$ gibt es unendlich viele mögliche Normalenvektoren.)*
"""

solution_3_7_b = r"""
### Lösung 3.7

---

**g)** Punkt $Q$:

$$Q = s\!\left(\tfrac{\pi}{2}\right) = \begin{pmatrix} \cos\tfrac{\pi}{2} \\ \sin\tfrac{\pi}{2} \\ \tfrac{\pi}{2} \end{pmatrix} = \begin{pmatrix} 0 \\ 1 \\ \tfrac{\pi}{2} \end{pmatrix}$$

---

**h)** Tangentenvektor und Tangente:

$$s'(t) = \begin{pmatrix} -\sin t \\ \cos t \\ 1 \end{pmatrix} \quad \Rightarrow \quad s'\!\left(\tfrac{\pi}{2}\right) = \begin{pmatrix} -1 \\ 0 \\ 1 \end{pmatrix}$$

Tangente in Parameterform:

$$\ell(s) = Q + s \cdot s'\!\left(\tfrac{\pi}{2}\right) = \begin{pmatrix} 0 \\ 1 \\ \tfrac{\pi}{2} \end{pmatrix} + s \begin{pmatrix} -1 \\ 0 \\ 1 \end{pmatrix}, \quad s \in \mathbb{R}$$

---

**i)** Normalenvektor $n = (a, b, c)^T \neq 0$ mit:

$$n \cdot (-1, 0, 1)^T = 0 \iff -a + c = 0 \iff c = a$$

Eine einfache Wahl ist z.B.:

$$n = \begin{pmatrix} 1 \\ 0 \\ 1 \end{pmatrix}$$

denn $(1, 0, 1) \cdot (-1, 0, 1) = -1 + 1 = 0$.  
*(Alternativ wäre auch $n = (0, 1, 0)^T$ möglich.)*
"""

# ════════════════════════════════════════════════════════════════
# Aufruf mit create_solution_toggle_3D
# ════════════════════════════════════════════════════════════════

display(create_solution_toggle_3D(
    problem_3_7_b,
    solution_3_7_b,
    solution_title="Lösung anzeigen / verbergen",
    plot_callback=plot_callback_3_7_b
))



<a id="tipps"></a>
## 💡 Lösungstipps

### Tipps zu Aufgabe 3.1
Leite $r(t)$ nach $t$ ab, um den Tangentialvektor $r'(t_0)$ zu erhalten. Der Normalenvektor steht senkrecht auf $r'(t_0)$: drehe einfach die Komponenten und ändere ein Vorzeichen. Bei **b)** prüfe zuerst, ob $|t|$ bei $t=0$ differenzierbar ist, bevor du $n(2)$ berechnest.

---

### Tipps zu Aufgabe 3.2
Wende die Produktregel auf $\|r'(t)\|^2 = \langle r'(t), r'(t)\rangle$ an und zeige, dass die Ableitung gleich $2\langle r', r''\rangle$ ist. Folgere daraus: konstante Geschwindigkeit bedeutet genau $\langle r', r''\rangle = 0$. Wende diese Aussage dann auf die drei Kurven $r_1, r_2, r_3$ an und berechne das Skalarprodukt jeweils explizit.

---

### Tipps zu Aufgabe 3.3
Berechne $r'(t)$ und setze $t_0=1$ ein, um Tangente und Normale als Parametergeraden zu schreiben. Für die Geradengleichung $y=mx+b$ der Tangente eliminiere den Parameter $s$ aus der Parameterform. Die Krümmung $\kappa(t)$ erhältst du direkt durch Einsetzen von $x'(t), x''(t), y'(t), y''(t)$ in die gegebene Formel.

---

### Tipps zu Aufgabe 3.4
Skizziere beide Geraden und erkenne, dass $\alpha$ die Bahn von $r_2$ horizontal verschiebt. Stelle das Gleichungssystem $r_1(t)=r_2(t)$ auf und löse es nach $t$ und $\alpha$ auf. Setze den gefundenen Zeitpunkt zurück ein, um den Treffpunkt $P$ zu bestimmen.

---

### Tipps zu Aufgabe 3.5
Setze $t=0$ für die Startpunkte ein und löse $y_i(t)=0$ für die Landezeitpunkte. Prüfe das Vorzeichen von $x'(t)$ bei beiden Raketen, um die Flugrichtung zu bestimmen. Löse dann $r_1(t)=r_2(t)$ als Gleichungssystem und prüfe, ob der Treffpunkt über dem Boden liegt.

---

### Tipps zu Aufgabe 3.6
Nutze bekannte trigonometrische bzw. hyperbolische Identitäten ($\cos^2t+\sin^2t=1$, $\cosh^2t-\sinh^2t=1$), um $t$ zu eliminieren. Bei Parabel und Kreis reicht direktes Einsetzen oder Umstellen nach $t$. Gib danach den Wertebereich von $x$ und $y$ an und skizziere die Kurve.

---

### Tipps zu Aufgabe 3.7
Berechne $r'(t)$ bzw. $s'(t)$ und setze den jeweiligen Parameterwert ein, um Tangenten- und Normalenvektor zu erhalten. Schreibe Tangente und Normale als Parametergeraden $P + s\cdot v$. In $\mathbb{R}^3$ gibt es unendlich viele Normalenvektoren — finde einen explizit durch die Bedingung $\langle n, s'(\tfrac{\pi}{2})\rangle = 0$.

---

---

## 📚 Weitere Ressourcen

- [Vorheriges Aufgabeblatt](./Mathematik_II_Uebung_02.ipynb)
- [Nächstes Aufgabeblatt](./Mathematik_II_Uebung_04.ipynb)
- [Lookup Notebook mit Funktionen](./Mathematik_II_Uebung.ipynb)

**Viel Erfolg beim Bearbeiten der Aufgaben!** 🎓